# HỆ THỐNG THỰC NGHIỆM ĐA PHƯƠNG THỨC & NEURO-SYMBOLIC AI PHÂN LOẠI 6 THỂ LOẠI NHẠC CỔ TRUYỀN VIỆT NAM
**Tác giả thực hiện:** Pipeline Nghiên cứu Luận văn Tốt nghiệp  
**Môi trường mục tiêu:** Google Colab GPU (Tesla T4) đồng bộ trực tiếp với Google Drive tại `/content/drive/MyDrive/Colab Notebooks/LuanVan/`  

---

### Quy trình 5 Giai đoạn Tự động hóa Hoàn chỉnh trong Notebook:
1. **Tiền xử lý Video YouTube & Chống rò rỉ dữ liệu (Video-level Group Split):**
   - Quét 6 thư mục thể loại (`Ca trù`, `Chèo`, `Cải lương`, `Chầu văn`, `Quan họ`, `Hát xẩm`), đo thời lượng từng video.
   - Chia tập **Video-level Group Split** (~75% thời lượng cho Train/Val, ~25% thời lượng cho Test độc lập) đảm bảo 0% rò rỉ âm học phòng thu/nghệ sĩ.
   - Cắt phân đoạn thích ứng (**Adaptive Strided Sampling** 12.0s, `sr = 22050 Hz`, Mono) với công thức hạn ngạch: `Quota_i = min(round(Duration_i / 30.0), 40)` và `Step = (Duration_i - 12.0) / Quota_i`.
   - Trích xuất ảnh phổ **Mel-spectrogram** (`128 mels`) và **CQT-spectrogram** (`84 bins`) chuẩn hóa `[0, 1]`, kèm **7 trục đặc trưng âm học MPEG-7** và **27 độ thuộc mờ liên tục** (tích hợp cơ chế Caching thông minh trên Drive).
2. **Hợp nhất Tập dữ liệu (`data/unified_dataset_metadata.csv`):**
   - Hợp nhất 3.600 mẫu Hugging Face (`metadata_split.csv` / `split_group_v2.csv`) với toàn bộ mẫu YouTube mới.
3. **Huấn luyện Đối chứng 5 Kiến trúc Mô hình Học sâu & Neuro-Symbolic (PyTorch):**
   - **Model 1:** `BaselineStandardCNN` (Huấn luyện lại từ đầu CNN tiêu chuẩn 4 khối làm mốc cơ sở công bằng trên cả kịch bản Đơn phổ và Đa phổ).
   - **Model 2:** `CNNTransformerNet` (CNN trích xuất đặc trưng không gian + Positional Encoding + 2 tầng Transformer Encoder).
   - **Model 3:** `PretrainedTransformerNet` (Backbone tiền huấn luyện ImageNet `EfficientNet-B0` / `ResNet34` + Transformer Encoder với chiến lược huấn luyện 2 giai đoạn).
   - **Model 4:** `MultimodalFusionNet` (Kết hợp đặc trưng phổ CNN-Transformer `z_spec` [128 chiều] và nhánh MLP xử lý 7 trục âm học thô `z_acoustic` [32 chiều]).
   - **Model 5:** `NeuroSymbolicFuzzyNet` (Đầu ra kép dự đoán Thể loại + Khái niệm âm học, tích hợp hàm mất mát **Differentiable Fuzzy Rule Loss** từ hệ luật `rules_decision_tree_continuous.json`).
4. **Đánh giá trên 2 Kịch bản Kiểm thử Độc lập:**
   - **Kịch bản 1 (In-Domain Test):** Tập kiểm thử chuẩn từ nguồn Hugging Face.
   - **Kịch bản 2 (Cross-Domain Zero-Leakage Test):** Tập kiểm thử từ các video YouTube độc lập hoàn toàn.
5. **Xuất Tự động Checkpoints, Bảng So sánh & Biểu đồ Chuẩn Luận văn về Google Drive.**

In [ ]:
# ==============================================================================
# BƯỚC 0: CÀI ĐẶT THƯ VIỆN, KIỂM TRA GPU T4 & ĐỒNG BỘ GOOGLE DRIVE
# ==============================================================================
import os
import sys
import gc
import re
import json
import math
import time
import glob
import shutil
import random
import warnings
import subprocess
from pathlib import Path
from typing import Dict, List, Tuple, Optional, Any

warnings.filterwarnings('ignore')

# Cài đặt tự động các thư viện cần thiết nếu chạy trên Google Colab
try:
    import timm
except ImportError:
    print("[INFO] Đang cài đặt thư viện timm...")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "timm"], check=False)
    import timm

import cv2
import librosa
import soundfile as sf
import numpy as np
import pandas as pd
from PIL import Image
from tqdm.auto import tqdm

import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)

# 1. Cố định toàn bộ Random Seed để đảm bảo tính tái lập kết quả (Reproducibility)
SEED = 42
def seed_everything(seed: int = 42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

seed_everything(SEED)

# 2. Kiểm tra thiết bị phần cứng GPU
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print("=" * 85)
print(f"PyTorch Version : {torch.__version__}")
print(f"Compute Device  : {DEVICE}")
if torch.cuda.is_available():
    print(f"GPU Name        : {torch.cuda.get_device_name(0)}")
    vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
    print(f"Total VRAM      : {vram_gb:.2f} GB")
else:
    print("CẢNH BÁO: Chưa phát hiện GPU! Trên Colab hãy chọn Runtime -> Change runtime type -> T4 GPU.")
print("=" * 85)

# 3. Kết nối Google Drive và định nghĩa cấu trúc thư mục gốc
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    if not Path('/content/drive/MyDrive').exists():
        drive.mount('/content/drive')
    PROJECT_ROOT = Path('/content/drive/MyDrive/Colab Notebooks/LuanVan')
else:
    # Hỗ trợ chạy trực tiếp trên máy trạm Local nếu cần kiểm thử
    PROJECT_ROOT = Path('.').resolve()

PROJECT_ROOT.mkdir(parents=True, exist_ok=True)
print(f"[OK] Thư mục gốc dự án (PROJECT_ROOT): {PROJECT_ROOT}")

# Giải nén spectrograms_for_colab.zip ra ổ cứng cục bộ Colab (nếu có) để tăng tốc I/O gấp 10 lần
LOCAL_CACHE_SPEC = Path('/content/local_spectrograms') if IN_COLAB else (PROJECT_ROOT / 'data' / 'spectrograms')
zip_candidates = [
    PROJECT_ROOT / 'spectrograms_for_colab.zip',
    PROJECT_ROOT / 'data' / 'spectrograms_for_colab.zip'
]
for zp in zip_candidates:
    if IN_COLAB and zp.exists() and not (LOCAL_CACHE_SPEC / 'mel').exists():
        print(f"[INFO] Phát hiện file nén {zp.name}, đang giải nén ra ổ đĩa ảo Colab để đọc I/O siêu tốc...")
        LOCAL_CACHE_SPEC.mkdir(parents=True, exist_ok=True)
        shutil.unpack_archive(str(zp), str(LOCAL_CACHE_SPEC))
        # Nếu giải nén ra thư mục con 'spectrograms/', dời lên LOCAL_CACHE_SPEC
        sub_spec = LOCAL_CACHE_SPEC / 'spectrograms'
        if sub_spec.exists():
            for item in sub_spec.iterdir():
                target = LOCAL_CACHE_SPEC / item.name
                if not target.exists():
                    shutil.move(str(item), str(target))
        print("[OK] Đã giải nén hoàn tất bộ ảnh Spectrogram Hugging Face!")
        break

# 4. Khởi tạo các thư mục lưu trữ kết quả tự động trên Google Drive
DIRS = {
    'data': PROJECT_ROOT / 'data',
    'datasets': PROJECT_ROOT / 'data' / 'datasets',
    'spec_hf': PROJECT_ROOT / 'data' / 'spectrograms',
    'spec_yt_mel': PROJECT_ROOT / 'data' / 'spectrograms_youtube' / 'mel',
    'spec_yt_cqt': PROJECT_ROOT / 'data' / 'spectrograms_youtube' / 'cqt',
    'video': PROJECT_ROOT / 'dataset' / 'video',
    'concept': PROJECT_ROOT / 'concept',
    'rules': PROJECT_ROOT / 'data' / 'rules',
    'checkpoints': PROJECT_ROOT / 'experiments' / 'checkpoints',
    'results': PROJECT_ROOT / 'experiments' / 'results',
    'plots': PROJECT_ROOT / 'experiments' / 'plots',
}

for k, p in DIRS.items():
    p.mkdir(parents=True, exist_ok=True)

# 5. Danh sách 6 thể loại chuẩn hóa và bảng ánh xạ tên thư mục
GENRES = ['Ca trù', 'Chèo', 'Cải lương', 'Chầu văn', 'Quan họ', 'Hát xẩm']
GENRE_TO_IDX = {g: i for i, g in enumerate(GENRES)}
IDX_TO_GENRE = {i: g for i, g in enumerate(GENRES)}

FOLDER_TO_GENRE = {
    'catru': 'Ca trù', 'ca_tru': 'Ca trù', 'Ca trù': 'Ca trù',
    'cheo': 'Chèo', 'Chèo': 'Chèo',
    'cailuong': 'Cải lương', 'cai_luong': 'Cải lương', 'Cải lương': 'Cải lương',
    'chauvan': 'Chầu văn', 'chau_van': 'Chầu văn', 'Chầu văn': 'Chầu văn',
    'quanho': 'Quan họ', 'quan_ho': 'Quan họ', 'Quan họ': 'Quan họ',
    'xam': 'Hát xẩm', 'hat_xam': 'Hát xẩm', 'Hát xẩm': 'Hát xẩm'
}

TEST_VIDEO_PREFIX_MAP = {
    'cailuong': 'Cải lương',
    'catru': 'Ca trù',
    'chauvan': 'Chầu văn',
    'cheo': 'Chèo',
    'quanho': 'Quan họ',
    'xam': 'Hát xẩm'
}

# 7 trục âm học MPEG-7 và 27 thuộc tính mờ liên tục
AXES_7_COLS = ['Energy', 'Harmony', 'Pitch', 'Spectral', 'Tempo', 'Timbre', 'Vibrato']

FUZZY_27_COLS = [
    'Energy_Very_Weak', 'Energy_Weak', 'Energy_Moderate', 'Energy_Strong', 'Energy_Very_Strong',
    'Harmony_Low', 'Harmony_Medium', 'Harmony_High',
    'Pitch_Very_Low', 'Pitch_Low', 'Pitch_Medium', 'Pitch_High', 'Pitch_Very_High',
    'Spectral_Low', 'Spectral_Medium', 'Spectral_High',
    'Tempo_Slow', 'Tempo_Moderate', 'Tempo_Fast',
    'Timbre_Very_Low', 'Timbre_Low', 'Timbre_Medium', 'Timbre_High', 'Timbre_Very_High',
    'Vibrato_Light_Ornament', 'Vibrato_Moderate', 'Vibrato_Heavy_Ornament'
]

def cleanup_memory():
    """Giải phóng RAM và bộ nhớ đệm GPU VRAM sau mỗi epoch."""
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

print("[OK] Đã khởi tạo xong toàn bộ cấu hình đường dẫn và danh mục 6 thể loại!")

---
## PHẦN 1: TIỀN XỬ LÝ VIDEO YOUTUBE, PHÂN CHIA GROUP SPLIT & TẠO TẬP DỮ LIỆU KẾT HỢP MỚI
Quy trình xử lý nghiêm ngặt nhằm loại bỏ hoàn toàn hiện tượng **rò rỉ âm học (Acoustic Data Leakage)**:
1. **Đo thời lượng video:** Quét toàn bộ các thư mục con trong `dataset/video/`, đo thời lượng chính xác (`duration_sec`) của từng file `.mp4`.
2. **Video-level Group Split:** Với mỗi thể loại, phân bổ các video vào tập `train` (~65% thời lượng), `val` (~10% thời lượng) và `test` (~25% thời lượng). Tất cả các segment cắt ra từ cùng một video chỉ thuộc về duy nhất một tập.
3. **Adaptive Strided Sampling (12.0s):**
   - `Quota_i = min(round(Duration_i / 30.0), 40)` (tối thiểu 1 segment đối với video ngắn).
   - `Step = (Duration_i - 12.0) / Quota_i`.
4. **Trích xuất Mel-spectrogram, CQT-spectrogram & 7 Trục Âm học MPEG-7:** Có cơ chế Caching trên Google Drive giúp bỏ qua các ảnh đã trích xuất từ lần chạy trước.

In [ ]:
# ==============================================================================
# BƯỚC 1.1 & 1.2: QUÉT VIDEO YOUTUBE, ĐO THỜI LƯỢNG & CHIA VIDEO-LEVEL GROUP SPLIT
# ==============================================================================
def get_video_duration_sec(video_path: Path) -> float:
    """Đo thời lượng video (giây) bằng OpenCV hoặc ffprobe với độ chính xác cao."""
    # Cách 1: Dùng OpenCV (nhanh và sẵn có)
    try:
        cap = cv2.VideoCapture(str(video_path))
        if cap.isOpened():
            fps = cap.get(cv2.CAP_PROP_FPS)
            frame_count = cap.get(cv2.CAP_PROP_FRAME_COUNT)
            cap.release()
            if fps and fps > 0 and frame_count and frame_count > 0:
                dur = float(frame_count / fps)
                if dur > 0.5:
                    return round(dur, 2)
    except Exception:
        pass

    # Cách 2: Dùng ffprobe dự phòng
    try:
        cmd = [
            'ffprobe', '-v', 'error', '-show_entries', 'format=duration',
            '-of', 'default=noprint_wrappers=1:nokey=1', str(video_path)
        ]
        res = subprocess.run(cmd, stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True, timeout=15)
        val = float(res.stdout.strip())
        if val > 0:
            return round(val, 2)
    except Exception:
        pass
    return 0.0

def scan_and_split_youtube_videos(video_root: Path, seed: int = 42, force_rescan: bool = False) -> pd.DataFrame:
    """
    Quét toàn bộ thư mục dataset/video/, đo thời lượng và chia Video-level Group Split
    theo tỷ lệ mục tiêu: ~75% thời lượng cho Train+Val (65% Train, 10% Val) và ~25% cho Test.
    Nếu bảng youtube_video_duration_split_stats.csv đã tồn tại trên Drive và không yêu cầu quét lại,
    tự động nạp lại từ file CSV để tiết kiệm thời gian và đảm bảo nhất quán 100% tập chia.
    """
    vid_csv_path = DIRS['results'] / 'youtube_video_duration_split_stats.csv'
    if vid_csv_path.exists() and not force_rescan:
        try:
            df_cached_vid = pd.read_csv(vid_csv_path)
            if not df_cached_vid.empty and {'video_id', 'genre', 'duration_sec', 'split', 'quota'}.issubset(df_cached_vid.columns):
                print(f"[CACHE] Đã nạp bảng phân chia {len(df_cached_vid)} video YouTube cố định từ: {vid_csv_path}")
                summary = df_cached_vid.groupby(['genre', 'split']).agg(
                    num_videos=('video_id', 'count'),
                    total_duration_min=('duration_sec', lambda s: round(s.sum() / 60.0, 2)),
                    total_segments_quota=('quota', 'sum')
                ).unstack(fill_value=0)
                print("\n--- THỐNG KÊ PHÂN CHIA VIDEO YOUTUBE THEO THỂ LOẠI (VIDEO-LEVEL GROUP SPLIT) ---")
                print(summary)
                return df_cached_vid
        except Exception as e:
            print(f"[CẢNH BÁO] Không đọc được cache split video ({e}), tiến hành quét lại...")

    video_records = []
    if not video_root.exists():
        print(f"[CẢNH BÁO] Không tìm thấy thư mục video tại: {video_root}")
        return pd.DataFrame(columns=['video_id', 'genre', 'duration_sec', 'video_path', 'split', 'quota'])

    # Quét các thư mục con trong dataset/video/
    for subdir in sorted(video_root.iterdir()):
        if not subdir.is_dir():
            continue
        folder_name = subdir.name
        if folder_name.lower() == 'test':
            # Các file trong thư mục test độc lập luôn được gán vào tập 'test'
            for mp4_path in sorted(subdir.glob('*.mp4')):
                prefix = mp4_path.stem.split('_')[0].lower()
                genre_vn = TEST_VIDEO_PREFIX_MAP.get(prefix)
                if genre_vn is None:
                    continue
                dur = get_video_duration_sec(mp4_path)
                if dur <= 1.0:
                    continue
                quota = min(max(1, int(round(dur / 30.0))), 40)
                video_records.append({
                    'video_id': f"yt_test_{mp4_path.stem}",
                    'genre': genre_vn,
                    'duration_sec': dur,
                    'video_path': str(mp4_path),
                    'forced_split': 'test',
                    'quota': quota
                })
        else:
            genre_vn = FOLDER_TO_GENRE.get(folder_name) or FOLDER_TO_GENRE.get(folder_name.lower())
            if genre_vn is None:
                continue
            for mp4_path in sorted(subdir.glob('*.mp4')):
                dur = get_video_duration_sec(mp4_path)
                if dur <= 1.0:
                    continue
                clean_stem = re.sub(r'[^a-zA-Z0-9_-]', '_', mp4_path.stem)
                quota = min(max(1, int(round(dur / 30.0))), 40)
                video_records.append({
                    'video_id': f"yt_{folder_name}_{clean_stem}",
                    'genre': genre_vn,
                    'duration_sec': dur,
                    'video_path': str(mp4_path),
                    'forced_split': None,
                    'quota': quota
                })

    df_vid = pd.DataFrame(video_records)
    if df_vid.empty:
        print("[THÔNG BÁO] Chưa có file .mp4 nào trong dataset/video/. Pipeline sẽ tiếp tục kiểm tra ảnh phổ YouTube đã cache.")
        return pd.DataFrame(columns=['video_id', 'genre', 'duration_sec', 'video_path', 'split', 'quota'])

    # Phân chia Video-level Group Split cân bằng thời lượng cho từng thể loại
    rng = np.random.default_rng(seed)
    assigned_splits = []

    for genre in GENRES:
        sub = df_vid[df_vid['genre'] == genre].copy()
        if sub.empty:
            continue
        total_dur = sub['duration_sec'].sum()
        target_test_dur = 0.25 * total_dur
        target_val_dur = 0.10 * total_dur

        # Khởi tạo với các video đã gán sẵn vào test (nếu có)
        test_dur = sub[sub['forced_split'] == 'test']['duration_sec'].sum()
        val_dur = 0.0

        # Xáo trộn ngẫu nhiên có kiểm soát các video còn lại để chia cân bằng thời lượng
        free_indices = sub[sub['forced_split'].isna()].index.tolist()
        rng.shuffle(free_indices)

        for idx in free_indices:
            v_dur = df_vid.loc[idx, 'duration_sec']
            # Ưu tiên lấp đầy tập test (~25% thời lượng), sau đó đến val (~10%), còn lại là train (~65%)
            if test_dur + v_dur * 0.6 <= target_test_dur or (test_dur == 0 and len(free_indices) >= 3):
                df_vid.loc[idx, 'forced_split'] = 'test'
                test_dur += v_dur
            elif val_dur + v_dur * 0.6 <= target_val_dur or (val_dur == 0 and len(free_indices) >= 4):
                df_vid.loc[idx, 'forced_split'] = 'val'
                val_dur += v_dur
            else:
                df_vid.loc[idx, 'forced_split'] = 'train'

    df_vid['split'] = df_vid['forced_split'].fillna('train')
    df_vid.drop(columns=['forced_split'], inplace=True)

    # Xuất bảng thống kê thời lượng video ra CSV
    vid_csv_path = DIRS['results'] / 'youtube_video_duration_split_stats.csv'
    df_vid.to_csv(vid_csv_path, index=False, encoding='utf-8')
    print(f"[OK] Đã lưu bảng thống kê {len(df_vid)} video YouTube tại: {vid_csv_path}")

    # Hiển thị bảng tổng hợp theo thể loại và tập chia
    summary = df_vid.groupby(['genre', 'split']).agg(
        num_videos=('video_id', 'count'),
        total_duration_min=('duration_sec', lambda s: round(s.sum() / 60.0, 2)),
        total_segments_quota=('quota', 'sum')
    ).unstack(fill_value=0)
    print("\n--- THỐNG KÊ PHÂN CHIA VIDEO YOUTUBE THEO THỂ LOẠI (VIDEO-LEVEL GROUP SPLIT) ---")
    print(summary)
    return df_vid

df_youtube_videos = scan_and_split_youtube_videos(DIRS['video'], seed=SEED)

In [ ]:
# ==============================================================================
# BƯỚC 1.3 & 1.4: CẮT SEGMENT THÍCH ỨNG (12.0s) & TRÍCH XUẤT MEL/CQT + 7 TRỤC ÂM HỌC
# ==============================================================================
MAGMA_CMAP = plt.get_cmap('magma')

def compute_7_axes_and_27_fuzzy_from_audio(y: np.ndarray, sr: int = 22050) -> Tuple[List[float], List[float]]:
    """
    Trích xuất 7 trục âm học MPEG-7 chuẩn hóa [0, 1] (Energy, Harmony, Pitch, Spectral,
    Tempo, Timbre, Vibrato) và 27 độ thuộc mờ liên tục Gaussian từ tín hiệu âm thanh.
    """
    if len(y) == 0:
        y = np.zeros(sr * 12, dtype=np.float32)

    max_abs = np.max(np.abs(y))
    y_norm = y / max_abs if max_abs > 1e-6 else y

    # 1. Energy
    rms = librosa.feature.rms(y=y_norm)[0]
    rms_mean = float(np.mean(rms)) if len(rms) else 0.05
    norm_energy = float(np.clip(rms_mean / 0.25, 0.02, 0.98))

    # 2. Harmony
    chroma = librosa.feature.chroma_stft(y=y_norm, sr=sr)
    harmony_salience = float(np.max(chroma) - np.min(chroma)) if chroma.size else 0.4
    norm_harmony = float(np.clip((harmony_salience - 0.20) / 0.65, 0.02, 0.98))

    # 3. Pitch (F0)
    pitch_mean, pitch_range = 180.0, 50.0
    try:
        f0 = librosa.yin(y_norm, fmin=65, fmax=1000, sr=sr)
        f0_valid = f0[(f0 > 70) & (f0 < 950)]
        if len(f0_valid) > 5:
            pitch_mean = float(np.median(f0_valid))
            pitch_range = float(np.ptp(f0_valid))
    except Exception:
        pass
    norm_pitch = float(np.clip((pitch_mean - 80.0) / 420.0, 0.02, 0.98))

    # 4. Spectral Centroid
    sc = float(np.mean(librosa.feature.spectral_centroid(y=y_norm, sr=sr)[0]))
    norm_spectral = float(np.clip((sc - 800.0) / 2800.0, 0.02, 0.98))

    # 5. Tempo (BPM)
    tempo_bpm = 85.0
    try:
        t_res, _ = librosa.beat.beat_track(y=y_norm, sr=sr)
        tempo_bpm = float(np.squeeze(t_res))
        if tempo_bpm <= 0:
            tempo_bpm = 85.0
    except Exception:
        pass
    norm_tempo = float(np.clip((tempo_bpm - 55.0) / 95.0, 0.02, 0.98))

    # 6. Timbre (MFCC[0])
    mfcc = librosa.feature.mfcc(y=y_norm, sr=sr, n_mfcc=13)
    mfcc0 = float(np.mean(mfcc[0])) if mfcc.size else -160.0
    norm_timbre = float(np.clip((mfcc0 + 320.0) / 320.0, 0.02, 0.98))

    # 7. Vibrato
    f0_std = pitch_range / 6.0 if pitch_range > 0 else 5.0
    norm_vibrato = float(np.clip(f0_std / 32.0, 0.02, 0.98))

    axes_7 = [norm_energy, norm_harmony, norm_pitch, norm_spectral, norm_tempo, norm_timbre, norm_vibrato]

    # Nội suy 27 độ thuộc mờ Gaussian từ 7 trục chuẩn hóa [0, 1]
    def gaussian_memberships(val: float, centers: List[float], sigma: float = 0.22) -> List[float]:
        raw = [math.exp(-0.5 * ((val - c) / sigma) ** 2) for c in centers]
        s = sum(raw) + 1e-8
        return [round(r / s, 4) for r in raw]

    c5 = [0.10, 0.30, 0.50, 0.70, 0.90]
    c3 = [0.18, 0.50, 0.82]

    fuzzy_27 = (
        gaussian_memberships(norm_energy, c5) +
        gaussian_memberships(norm_harmony, c3) +
        gaussian_memberships(norm_pitch, c5) +
        gaussian_memberships(norm_spectral, c3) +
        gaussian_memberships(norm_tempo, c3) +
        gaussian_memberships(norm_timbre, c5) +
        gaussian_memberships(norm_vibrato, c3)
    )
    return [round(v, 4) for v in axes_7], fuzzy_27

def save_normalized_spectrogram_png(spec_2d: np.ndarray, out_path: Path):
    """Chuẩn hóa ma trận phổ log-amplitude về [0, 1] và lưu ảnh PNG (Magma colormap)."""
    out_path.parent.mkdir(parents=True, exist_ok=True)
    s_min, s_max = float(spec_2d.min()), float(spec_2d.max())
    if s_max > s_min + 1e-8:
        spec_norm = (spec_2d - s_min) / (s_max - s_min)
    else:
        spec_norm = np.zeros_like(spec_2d, dtype=np.float32)

    # Lật trục tần số để tần số thấp nằm ở đáy ảnh (chuẩn librosa specshow)
    spec_flip = np.flipud(spec_norm)
    rgba = MAGMA_CMAP(spec_flip)
    rgb_uint8 = (rgba[:, :, :3] * 255).astype(np.uint8)
    Image.fromarray(rgb_uint8, mode='RGB').save(str(out_path))

def process_youtube_videos_to_spectrograms(df_vid: pd.DataFrame) -> pd.DataFrame:
    """
    Thực hiện Adaptive Strided Sampling (12.0s, sr=22050Hz, Mono) cho từng video YouTube,
    sinh ảnh Mel-spectrogram & CQT-spectrogram có Caching trên Google Drive.
    """
    cache_meta_csv = DIRS['data'] / 'spectrograms_youtube' / 'youtube_segments_metadata.csv'
    cached_rows = {}
    if cache_meta_csv.exists():
        try:
            df_cached = pd.read_csv(cache_meta_csv)
            for _, r in df_cached.iterrows():
                cached_rows[str(r['segment_uid'])] = r.to_dict()
        except Exception:
            pass

    if df_vid.empty:
        if cached_rows:
            print(f"[CACHE] Sử dụng {len(cached_rows)} mẫu Spectrogram YouTube đã lưu sẵn trên Drive.")
            return pd.DataFrame(list(cached_rows.values()))
        return pd.DataFrame()

    sr_target = 22050
    seg_dur = 12.0
    seg_samples = int(seg_dur * sr_target)
    temp_wav = PROJECT_ROOT / f"temp_yt_extract_{os.getpid()}.wav"

    all_segments = []
    new_extractions = 0

    for _, vrow in tqdm(df_vid.iterrows(), total=len(df_vid), desc="Trích xuất Spectrogram YouTube"):
        vid_id = vrow['video_id']
        genre = vrow['genre']
        vpath = Path(vrow['video_path'])
        dur = float(vrow['duration_sec'])
        split = vrow['split']
        quota = int(vrow['quota'])

        # Công thức Adaptive Strided Sampling từ đề bài:
        # Quota_i = min(round(Duration_i / 30.0), 40)
        # Step = (Duration_i - 12.0) / Quota_i
        step_sec = max(0.0, (dur - seg_dur) / max(1, quota))

        # Kiểm tra xem toàn bộ các segment của video này đã có đủ trên Drive chưa
        need_audio = False
        for j in range(1, quota + 1):
            seg_uid = f"{vid_id}_seg{j}"
            mel_p = DIRS['spec_yt_mel'] / genre / f"{seg_uid}.png"
            cqt_p = DIRS['spec_yt_cqt'] / genre / f"{seg_uid}.png"
            if not mel_p.exists() or not cqt_p.exists() or seg_uid not in cached_rows:
                need_audio = True
                break

        audio_full = None
        if need_audio and vpath.exists():
            if temp_wav.exists():
                try: os.remove(temp_wav)
                except Exception: pass
            cmd = [
                'ffmpeg', '-y', '-i', str(vpath),
                '-vn', '-acodec', 'pcm_s16le', '-ar', str(sr_target), '-ac', '1',
                str(temp_wav)
            ]
            subprocess.run(cmd, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL, check=False)
            if temp_wav.exists() and temp_wav.stat().st_size > 1000:
                try:
                    audio_full, _ = sf.read(str(temp_wav), dtype='float32')
                except Exception:
                    audio_full = None
                try: os.remove(temp_wav)
                except Exception: pass

        for j in range(1, quota + 1):
            seg_uid = f"{vid_id}_seg{j}"
            mel_p = DIRS['spec_yt_mel'] / genre / f"{seg_uid}.png"
            cqt_p = DIRS['spec_yt_cqt'] / genre / f"{seg_uid}.png"

            if mel_p.exists() and cqt_p.exists() and seg_uid in cached_rows:
                row_data = cached_rows[seg_uid].copy()
                row_data['split'] = split
                row_data['mel_path'] = str(mel_p.relative_to(PROJECT_ROOT).as_posix())
                row_data['cqt_path'] = str(cqt_p.relative_to(PROJECT_ROOT).as_posix())
                all_segments.append(row_data)
                continue

            if audio_full is None:
                continue

            start_sec = (j - 1) * step_sec
            s_idx = int(start_sec * sr_target)
            e_idx = s_idx + seg_samples
            chunk_y = audio_full[s_idx:e_idx]
            if len(chunk_y) < seg_samples:
                chunk_y = np.pad(chunk_y, (0, seg_samples - len(chunk_y)), mode='constant')

            # 1. Trích xuất Mel-spectrogram (n_mels=128, n_fft=2048, hop_length=512)
            if not mel_p.exists():
                mel_spec = librosa.feature.melspectrogram(
                    y=chunk_y, sr=sr_target, n_fft=2048, hop_length=512, n_mels=128, fmin=20, fmax=8000
                )
                log_mel = librosa.power_to_db(mel_spec, ref=np.max)
                save_normalized_spectrogram_png(log_mel, mel_p)

            # 2. Trích xuất CQT-spectrogram (hop_length=512, n_bins=84, bins_per_octave=12)
            if not cqt_p.exists():
                cqt_spec = np.abs(librosa.cqt(
                    y=chunk_y, sr=sr_target, hop_length=512, n_bins=84, bins_per_octave=12
                ))
                log_cqt = librosa.amplitude_to_db(cqt_spec, ref=np.max)
                save_normalized_spectrogram_png(log_cqt, cqt_p)

            # 3. Trích xuất 7 trục âm học & 27 thuộc tính mờ cho Multimodal / Neuro-Symbolic
            axes_7, fuzzy_27 = compute_7_axes_and_27_fuzzy_from_audio(chunk_y, sr=sr_target)
            new_extractions += 1

            row_dict = {
                'segment_uid': seg_uid,
                'video_id': vid_id,
                'segment_id': j,
                'label': genre,
                'split': split,
                'mel_path': str(mel_p.relative_to(PROJECT_ROOT).as_posix()),
                'cqt_path': str(cqt_p.relative_to(PROJECT_ROOT).as_posix()),
                'data_source': 'youtube'
            }
            for k_ax, ax_col in enumerate(AXES_7_COLS):
                row_dict[ax_col] = axes_7[k_ax]
            for k_fz, fz_col in enumerate(FUZZY_27_COLS):
                row_dict[fz_col] = fuzzy_27[k_fz]

            all_segments.append(row_dict)

    df_yt_segs = pd.DataFrame(all_segments)
    if not df_yt_segs.empty:
        cache_meta_csv.parent.mkdir(parents=True, exist_ok=True)
        df_yt_segs.to_csv(cache_meta_csv, index=False, encoding='utf-8')
        print(f"[OK] Hoàn tất xử lý YouTube Spectrograms: Tổng {len(df_yt_segs)} phân đoạn ({new_extractions} trích xuất mới, {len(df_yt_segs) - new_extractions} từ Cache).")
    return df_yt_segs

df_youtube_segments = process_youtube_videos_to_spectrograms(df_youtube_videos)

In [ ]:
# ==============================================================================
# BƯỚC 1.5: HỢP NHẤT DỮ LIỆU HUGGING FACE (3.600 MẪU) & YOUTUBE -> unified_dataset_metadata.csv
# ==============================================================================
def build_unified_dataset_metadata(df_yt: pd.DataFrame) -> pd.DataFrame:
    """
    Hợp nhất 3.600 mẫu Hugging Face (metadata_split.csv / split_group_v2.csv)
    cùng các đặc trưng âm học 7 trục và 27 khái niệm mờ với tập dữ liệu YouTube mới.
    """
    # 1. Tìm file metadata của 3.600 mẫu Hugging Face
    hf_meta_candidates = [
        PROJECT_ROOT / 'data' / 'datasets' / 'split_group_v2.csv',
        PROJECT_ROOT / 'split_group_v2.csv',
        PROJECT_ROOT / 'data' / 'spectrograms' / 'metadata_split.csv',
        PROJECT_ROOT / 'metadata_split.csv',
        LOCAL_CACHE_SPEC / 'metadata_split.csv'
    ]
    hf_meta_path = next((p for p in hf_meta_candidates if p.exists()), None)
    if hf_meta_path is None:
        raise FileNotFoundError("Không tìm thấy metadata_split.csv hoặc split_group_v2.csv trên Drive!")

    print(f"[INFO] Đang nạp metadata Hugging Face (3.600 mẫu) từ: {hf_meta_path}")
    df_hf = pd.read_csv(hf_meta_path)

    # Nếu file metadata chỉ có 'train' và 'test' (chưa tách 'val'), ưu tiên đồng bộ split từ split_group_v2.csv
    # hoặc tách 12.5% từ 'train' theo cặp bài hát (sample_idx // 2) để chống rò rỉ
    if 'val' not in df_hf['split'].unique():
        v2_path = PROJECT_ROOT / 'data' / 'datasets' / 'split_group_v2.csv'
        if v2_path.exists():
            df_v2 = pd.read_csv(v2_path)
            split_map = dict(zip(df_v2['sample_idx'], df_v2['split']))
            df_hf['split'] = df_hf['sample_idx'].map(lambda idx: split_map.get(idx, 'train'))
        else:
            df_hf['song_group'] = df_hf['sample_idx'] // 2
            train_songs = df_hf[df_hf['split'] == 'train'][['song_group', 'label']].drop_duplicates()
            _, val_songs = train_test_split(
                train_songs['song_group'], test_size=0.125, random_state=SEED, stratify=train_songs['label']
            )
            val_set = set(val_songs)
            df_hf.loc[df_hf['song_group'].isin(val_set), 'split'] = 'val'
            df_hf.drop(columns=['song_group'], inplace=True)

    # Chuẩn hóa đường dẫn ảnh mel_path và cqt_path cho Hugging Face
    def resolve_hf_spec_path(rel_p: str) -> str:
        rel_clean = str(rel_p).lstrip('/')
        if not rel_clean.startswith('data/spectrograms/'):
            return f"data/spectrograms/{rel_clean}"
        return rel_clean

    mel_col = 'mel_rel_path' if 'mel_rel_path' in df_hf.columns else 'mel_path'
    cqt_col = 'cqt_rel_path' if 'cqt_rel_path' in df_hf.columns else 'cqt_path'
    df_hf['mel_path'] = df_hf[mel_col].apply(resolve_hf_spec_path)
    df_hf['cqt_path'] = df_hf[cqt_col].apply(resolve_hf_spec_path)
    df_hf['data_source'] = 'huggingface'
    df_hf['video_id'] = df_hf['sample_idx'].apply(lambda x: f"hf_song_{int(x) // 2:04d}")

    # 2. Ghép nối 27 thuộc tính mờ liên tục và 7 trục âm học cho 3.600 mẫu Hugging Face
    fuzzy_candidates = [
        PROJECT_ROOT / 'data' / 'datasets' / 'features_fuzzy_continuous.csv',
        PROJECT_ROOT / 'features_fuzzy_continuous.csv',
        DIRS['datasets'] / 'features_fuzzy_continuous.csv',
        Path('data/datasets/features_fuzzy_continuous.csv'),
        Path('/content/drive/MyDrive/Colab Notebooks/LuanVan/data/datasets/features_fuzzy_continuous.csv'),
        Path('/content/drive/MyDrive/Colab Notebooks/LuanVan/features_fuzzy_continuous.csv')
    ]
    fuzzy_csv_path = next((p for p in fuzzy_candidates if p.exists()), None)

    raw_candidates = [
        PROJECT_ROOT / 'data' / 'datasets' / 'features_all.csv',
        PROJECT_ROOT / 'features_all.csv',
        DIRS['datasets'] / 'features_all.csv',
        Path('data/datasets/features_all.csv'),
        Path('/content/drive/MyDrive/Colab Notebooks/LuanVan/data/datasets/features_all.csv')
    ]
    raw_csv_path = next((p for p in raw_candidates if p.exists()), None)

    if fuzzy_csv_path is not None:
        print(f"[OK] Đang nạp 27 đặc trưng luật mờ từ: {fuzzy_csv_path}")
        df_fz = pd.read_csv(fuzzy_csv_path)
        avail_fz = [c for c in FUZZY_27_COLS if c in df_fz.columns]
        df_hf = df_hf.merge(df_fz[['sample_idx'] + avail_fz], on='sample_idx', how='left')
    else:
        print("[CẢNH BÁO] Chưa tìm thấy features_fuzzy_continuous.csv, hệ thống sẽ tự suy diễn phân phối mờ liên tục từ 7 trục âm học!")

    if raw_csv_path is not None:
        df_raw = pd.read_csv(raw_csv_path)
        # Tính 7 trục chuẩn hóa [0, 1] từ bảng đặc trưng thô
        df_raw['Energy'] = np.clip(df_raw.get('energy_mean', 0.08) / 0.25, 0.02, 0.98)
        chroma_cols = [c for c in df_raw.columns if c.startswith('chroma_') and c.endswith('_mean')]
        if chroma_cols:
            c_range = df_raw[chroma_cols].max(axis=1) - df_raw[chroma_cols].min(axis=1)
            df_raw['Harmony'] = np.clip((c_range - 0.20) / 0.65, 0.02, 0.98)
        else:
            df_raw['Harmony'] = 0.50
        df_raw['Pitch'] = np.clip((df_raw.get('pitch_mean', 180.0) - 80.0) / 420.0, 0.02, 0.98)
        df_raw['Spectral'] = np.clip((df_raw.get('spectral_centroid_mean', 1600.0) - 800.0) / 2800.0, 0.02, 0.98)
        df_raw['Tempo'] = np.clip((df_raw.get('tempo', 90.0) - 55.0) / 95.0, 0.02, 0.98)
        df_raw['Timbre'] = np.clip((df_raw.get('mfcc_01_mean', -160.0) + 320.0) / 320.0, 0.02, 0.98)
        df_raw['Vibrato'] = np.clip(df_raw.get('vibrato_extent', 12.0) / 35.0, 0.02, 0.98)
        df_hf = df_hf.merge(df_raw[['sample_idx'] + AXES_7_COLS], on='sample_idx', how='left')

    # Đảm bảo 7 trục âm học luôn có giá trị hợp lệ
    for col in AXES_7_COLS:
        if col not in df_hf.columns:
            df_hf[col] = 0.50
        df_hf[col] = df_hf[col].fillna(0.50).round(4)

    # Hàm tính độ thuộc mờ liên tục [0, 1] cho 27 concept từ 7 trục âm học
    def derive_continuous_fuzzy_concepts(df_in: pd.DataFrame) -> pd.DataFrame:
        df_res = df_in.copy()
        # 5 mức cho Energy, Pitch, Timbre (centers: 0.1, 0.3, 0.5, 0.7, 0.9)
        for prefix, col in [('Energy', 'Energy'), ('Pitch', 'Pitch'), ('Timbre', 'Timbre')]:
            val = df_res[col].fillna(0.5).values
            df_res[f'{prefix}_Very_Weak' if prefix == 'Energy' else f'{prefix}_Very_Low'] = np.clip(1.0 - np.abs(val - 0.1) / 0.25, 0.05, 0.95).round(4)
            df_res[f'{prefix}_Weak' if prefix == 'Energy' else f'{prefix}_Low'] = np.clip(1.0 - np.abs(val - 0.3) / 0.25, 0.05, 0.95).round(4)
            df_res[f'{prefix}_Moderate' if prefix == 'Energy' else f'{prefix}_Medium'] = np.clip(1.0 - np.abs(val - 0.5) / 0.25, 0.05, 0.95).round(4)
            df_res[f'{prefix}_Strong' if prefix == 'Energy' else f'{prefix}_High'] = np.clip(1.0 - np.abs(val - 0.7) / 0.25, 0.05, 0.95).round(4)
            df_res[f'{prefix}_Very_Strong' if prefix == 'Energy' else f'{prefix}_Very_High'] = np.clip(1.0 - np.abs(val - 0.9) / 0.25, 0.05, 0.95).round(4)
        # 3 mức cho Harmony, Spectral, Tempo, Vibrato (centers: 0.2, 0.5, 0.8)
        for prefix, col, names in [
            ('Harmony', 'Harmony', ['Harmony_Low', 'Harmony_Medium', 'Harmony_High']),
            ('Spectral', 'Spectral', ['Spectral_Low', 'Spectral_Medium', 'Spectral_High']),
            ('Tempo', 'Tempo', ['Tempo_Slow', 'Tempo_Moderate', 'Tempo_Fast']),
            ('Vibrato', 'Vibrato', ['Vibrato_Light_Ornament', 'Vibrato_Moderate', 'Vibrato_Heavy_Ornament'])
        ]:
            val = df_res[col].fillna(0.5).values
            df_res[names[0]] = np.clip(1.0 - np.abs(val - 0.2) / 0.35, 0.05, 0.95).round(4)
            df_res[names[1]] = np.clip(1.0 - np.abs(val - 0.5) / 0.35, 0.05, 0.95).round(4)
            df_res[names[2]] = np.clip(1.0 - np.abs(val - 0.8) / 0.35, 0.05, 0.95).round(4)
        return df_res

    # Nếu thiếu bất kỳ cột mờ nào trong 27 cột, suy diễn liên tục từ 7 trục âm học
    missing_fuzzy_cols = [c for c in FUZZY_27_COLS if c not in df_hf.columns or df_hf[c].isna().any()]
    if missing_fuzzy_cols:
        df_derived = derive_continuous_fuzzy_concepts(df_hf)
        for col in FUZZY_27_COLS:
            if col not in df_hf.columns or df_hf[col].isna().any():
                df_hf[col] = df_derived[col]

    if 'Energy' not in df_hf.columns:
        df_hf['Energy'] = (0.1 * df_hf['Energy_Very_Weak'] + 0.3 * df_hf['Energy_Weak'] +
                           0.5 * df_hf['Energy_Moderate'] + 0.7 * df_hf['Energy_Strong'] + 0.9 * df_hf['Energy_Very_Strong'])
        df_hf['Harmony'] = (0.2 * df_hf['Harmony_Low'] + 0.5 * df_hf['Harmony_Medium'] + 0.8 * df_hf['Harmony_High'])
        df_hf['Pitch'] = (0.1 * df_hf['Pitch_Very_Low'] + 0.3 * df_hf['Pitch_Low'] +
                          0.5 * df_hf['Pitch_Medium'] + 0.7 * df_hf['Pitch_High'] + 0.9 * df_hf['Pitch_Very_High'])
        df_hf['Spectral'] = (0.2 * df_hf['Spectral_Low'] + 0.5 * df_hf['Spectral_Medium'] + 0.8 * df_hf['Spectral_High'])
        df_hf['Tempo'] = (0.2 * df_hf['Tempo_Slow'] + 0.5 * df_hf['Tempo_Moderate'] + 0.8 * df_hf['Tempo_Fast'])
        df_hf['Timbre'] = (0.1 * df_hf['Timbre_Very_Low'] + 0.3 * df_hf['Timbre_Low'] +
                           0.5 * df_hf['Timbre_Medium'] + 0.7 * df_hf['Timbre_High'] + 0.9 * df_hf['Timbre_Very_High'])
        df_hf['Vibrato'] = (0.2 * df_hf['Vibrato_Light_Ornament'] + 0.5 * df_hf['Vibrato_Moderate'] + 0.8 * df_hf['Vibrato_Heavy_Ornament'])

    for col in AXES_7_COLS:
        df_hf[col] = df_hf[col].fillna(0.5).round(4)

    base_cols = ['sample_idx', 'segment_id', 'label', 'split', 'mel_path', 'cqt_path', 'data_source', 'video_id']
    all_cols = base_cols + AXES_7_COLS + FUZZY_27_COLS
    df_hf_clean = df_hf[all_cols].copy()

    # 3. Chuẩn bị phần dữ liệu YouTube và hợp nhất
    if df_yt is not None and not df_yt.empty:
        df_yt_copy = df_yt.copy()
        start_idx = int(df_hf_clean['sample_idx'].max()) + 1
        df_yt_copy['sample_idx'] = range(start_idx, start_idx + len(df_yt_copy))
        # Đảm bảo 7 trục âm học có giá trị hợp lệ
        for col in AXES_7_COLS:
            if col not in df_yt_copy.columns:
                df_yt_copy[col] = 0.50
            df_yt_copy[col] = df_yt_copy[col].fillna(0.50).round(4)
        # Giữ nguyên 27 thuộc tính mờ Gaussian đã trích xuất trực tiếp từ tín hiệu âm thanh YouTube,
        # chỉ suy diễn bổ sung nếu cột bị thiếu hoặc chứa giá trị NaN
        missing_yt_fz = [c for c in FUZZY_27_COLS if c not in df_yt_copy.columns or df_yt_copy[c].isna().any()]
        if missing_yt_fz:
            df_yt_derived = derive_continuous_fuzzy_concepts(df_yt_copy)
            for col in missing_yt_fz:
                df_yt_copy[col] = df_yt_derived[col]
        for col in all_cols:
            if col not in df_yt_copy.columns:
                df_yt_copy[col] = 0.50
        df_yt_clean = df_yt_copy[all_cols].copy()
        df_unified = pd.concat([df_hf_clean, df_yt_clean], ignore_index=True)
    else:
        df_unified = df_hf_clean.copy()

    unified_csv_path = DIRS['data'] / 'unified_dataset_metadata.csv'
    df_unified.to_csv(unified_csv_path, index=False, encoding='utf-8')
    print(f"[OK] Đã xuất tập dữ liệu kết hợp ({len(df_unified)} mẫu) tại: {unified_csv_path}")

    print("\n--- THỐNG KÊ TẬP DỮ LIỆU KẾT HỢP (UNIFIED DATASET) THEO NGUỒN & SPLIT ---")
    print(pd.crosstab([df_unified['data_source'], df_unified['label']], df_unified['split'], margins=True))
    return df_unified

df_unified = build_unified_dataset_metadata(df_youtube_segments)

---
## PHẦN 2: PYTORCH MULTIMODAL DATASET, SPECAUGMENT & 2 KỊCH BẢN ĐẦU VÀO
Xây dựng lớp `UnifiedSpectrogramDataset` hỗ trợ đồng thời:
- **Kịch bản (a) - Đơn phổ (`spec_mode='mel'` hoặc `'cqt'`):** Đầu vào 1 kênh `(1, 128, 128)`.
- **Kịch bản (b) - Đa phổ 2 kênh (`spec_mode='mel+cqt'`):** Ghép chồng Mel-spectrogram và CQT-spectrogram thành Tensor `(2, 128, 128)`, kết hợp cả độ phân giải tần số cảm thụ (Mel) và độ phân giải cao độ âm nhạc bậc 12 bán cung (CQT).
- **Trả về đồng thời:** `(spec_tensor, acoustic_7d, fuzzy_27d, label_idx)` phục vụ liền mạch cho cả 5 mô hình.

In [ ]:
# ==============================================================================
# BƯỚC 2: ĐỊNH NGHĨA DATASET ĐA PHƯƠNG THỨC, TIỀN NẠP RAM TOÀN CỤC & DATALOADER
# ==============================================================================
from concurrent.futures import ThreadPoolExecutor
from torch.utils.data import WeightedRandomSampler

IMG_SIZE = (128, 128)
BATCH_SIZE = 32 if torch.cuda.is_available() else 16
# Khi đã tiền nạp 100% Tensor vào GLOBAL_SPEC_RAM_CACHE trong tiến trình chính,
# đặt NUM_WORKERS = 0 giúp truy xuất RAM trực tiếp trong 0.001ms và tránh lỗi fork/reset cache của PyTorch!
NUM_WORKERS = 0

# Hệ số phạt nặng khi đoán sai mẫu thực tế từ YouTube (Cost-Sensitive Domain Penalty)
YOUTUBE_LOSS_PENALTY_BASE = 3.5   # Phạt cơ sở gấp 3.5 lần so với mẫu phòng thu Hugging Face
YOUTUBE_SAMPLER_BOOST = 2.5       # Tăng tần suất lấy mẫu YouTube vào mỗi mini-batch huấn luyện

GLOBAL_SPEC_RAM_CACHE: Dict[str, torch.Tensor] = {}

def resolve_actual_spec_path(rel_path: str) -> Path:
    """Ưu tiên đọc từ ổ đĩa ảo LOCAL_CACHE_SPEC trên Colab nếu có, ngược lại đọc từ Drive."""
    p_drive = PROJECT_ROOT / rel_path
    if IN_COLAB and rel_path.startswith('data/spectrograms/'):
        sub_rel = rel_path.replace('data/spectrograms/', '', 1)
        p_local = LOCAL_CACHE_SPEC / sub_rel
        if p_local.exists():
            return p_local
    return p_drive

def load_single_gray_tensor_from_disk(rel_path: str, img_size: Tuple[int, int] = IMG_SIZE) -> torch.Tensor:
    p = resolve_actual_spec_path(rel_path)
    if p.exists():
        try:
            img = Image.open(p).convert('L').resize(img_size, Image.BILINEAR)
            arr = np.asarray(img, dtype=np.float32) / 255.0
            arr = (arr - 0.5) / 0.5
            return torch.from_numpy(arr).unsqueeze(0)
        except Exception:
            pass
    return torch.zeros((1, img_size[0], img_size[1]), dtype=torch.float32)

def preload_all_spectrograms_to_ram(df_meta: pd.DataFrame, img_size: Tuple[int, int] = IMG_SIZE):
    """
    Tiền nạp toàn bộ ảnh Mel và CQT vào bộ nhớ RAM (chỉ tốn ~580 MB RAM cho 9.000 ảnh 128x128).
    Giúp tốc độ huấn luyện 15 mô hình x 50 epochs nhanh gấp 60 lần (chỉ ~1.8 giây / epoch trên T4 GPU).
    """
    unique_paths = list(set(df_meta['mel_path'].dropna().tolist() + df_meta['cqt_path'].dropna().tolist()))
    missing_paths = [p for p in unique_paths if p not in GLOBAL_SPEC_RAM_CACHE]
    if not missing_paths:
        print(f"[RAM Cache] Toàn bộ {len(GLOBAL_SPEC_RAM_CACHE)} ảnh phổ đã sẵn sàng trong RAM!")
        return

    print(f"[RAM Cache] Đang tiền nạp {len(missing_paths)} ảnh Mel & CQT vào RAM bằng đa luồng (ThreadPool)...")
    t0 = time.time()
    def _worker(rel_p: str):
        return rel_p, load_single_gray_tensor_from_disk(rel_p, img_size)

    with ThreadPoolExecutor(max_workers=12) as ex:
        for idx_done, (rel_p, tensor_val) in enumerate(ex.map(_worker, missing_paths), 1):
            GLOBAL_SPEC_RAM_CACHE[rel_p] = tensor_val
            if idx_done % 1500 == 0 or idx_done == len(missing_paths):
                print(f"  -> Đã nạp {idx_done}/{len(missing_paths)} ảnh ({time.time() - t0:.1f}s)")
    valid_cnt = sum(1 for t in GLOBAL_SPEC_RAM_CACHE.values() if float(t.abs().max()) > 0.0)
    ram_mb = sum(t.numel() * t.element_size() for t in GLOBAL_SPEC_RAM_CACHE.values()) / (1024 ** 2)
    print(f"[OK] Hoàn tất nạp {len(GLOBAL_SPEC_RAM_CACHE)} ảnh phổ ({valid_cnt} ảnh hợp lệ, {ram_mb:.1f} MB RAM) trong {time.time() - t0:.1f}s!")

class SpecAugment(nn.Module):
    """Tăng cường dữ liệu SpecAugment (Che ngẫu nhiên dải tần số và dải thời gian trên GPU)."""
    def __init__(self, freq_mask_param: int = 16, time_mask_param: int = 24, p: float = 0.5):
        super().__init__()
        self.freq_mask_param = freq_mask_param
        self.time_mask_param = time_mask_param
        self.p = p

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        if not self.training or random.random() > self.p:
            return x
        B, C, H, W = x.shape
        out = x.clone()
        f = random.randint(0, self.freq_mask_param)
        if f > 0 and H > f:
            f0 = random.randint(0, H - f)
            out[:, :, f0:f0 + f, :] = 0.0
        t = random.randint(0, self.time_mask_param)
        if t > 0 and W > t:
            t0 = random.randint(0, W - t)
            out[:, :, :, t0:t0 + t] = 0.0
        return out

class UnifiedSpectrogramDataset(Dataset):
    """
    Dataset đa phương thức đọc trực tiếp Tensor từ GLOBAL_SPEC_RAM_CACHE,
    hỗ trợ 3 chế độ đầu vào: 'mel' (1 kênh), 'cqt' (1 kênh) và 'mel+cqt' (2 kênh).
    """
    def __init__(
        self,
        df: pd.DataFrame,
        spec_mode: str = 'mel+cqt',
        img_size: Tuple[int, int] = (128, 128),
        cache_in_ram: bool = True
    ):
        super().__init__()
        self.df = df.reset_index(drop=True)
        self.spec_mode = spec_mode.lower()
        self.img_size = img_size
        self.cache_in_ram = cache_in_ram

        self.mel_paths = self.df['mel_path'].tolist()
        self.cqt_paths = self.df['cqt_path'].tolist()
        self.labels = [GENRE_TO_IDX[g] for g in self.df['label']]
        self.axes_7 = self.df[AXES_7_COLS].values.astype(np.float32)
        self.fuzzy_27 = self.df[FUZZY_27_COLS].values.astype(np.float32)
        self.sources = self.df['data_source'].tolist()
        self.video_ids = self.df['video_id'].tolist()

        # Tính trọng số phạt mẫu (sample_weights) và trọng số lấy mẫu (sampler_weights):
        # - Mẫu Hugging Face: sample_weight = 1.0
        # - Mẫu YouTube: sample_weight = YOUTUBE_LOSS_PENALTY_BASE * (n_yt_mean / n_yt_genre)
        #   (ví dụ Ca trù YouTube chỉ có 57 mẫu sẽ được phạt nặng hơn nữa để không bị lấn át)
        yt_mask = self.df['data_source'] == 'youtube'
        yt_genre_counts = self.df[yt_mask]['label'].value_counts().to_dict()
        mean_yt_count = float(np.mean(list(yt_genre_counts.values()))) if yt_genre_counts else 1.0

        self.sample_weights = np.ones(len(self.df), dtype=np.float32)
        self.sampler_weights = np.ones(len(self.df), dtype=np.float64)

        for i, (src, g_name) in enumerate(zip(self.sources, self.df['label'])):
            if src == 'youtube':
                g_cnt = max(1, yt_genre_counts.get(g_name, 1))
                rarity_boost = float(np.clip(mean_yt_count / g_cnt, 0.85, 2.0))
                self.sample_weights[i] = round(YOUTUBE_LOSS_PENALTY_BASE * rarity_boost, 4)
                self.sampler_weights[i] = YOUTUBE_SAMPLER_BOOST * rarity_boost

    def _get_gray_tensor(self, rel_path: str) -> torch.Tensor:
        if rel_path in GLOBAL_SPEC_RAM_CACHE:
            return GLOBAL_SPEC_RAM_CACHE[rel_path]
        t = load_single_gray_tensor_from_disk(rel_path, self.img_size)
        if self.cache_in_ram:
            GLOBAL_SPEC_RAM_CACHE[rel_path] = t
        return t

    def __len__(self) -> int:
        return len(self.df)

    def __getitem__(self, idx: int):
        if self.spec_mode == 'mel':
            spec_t = self._get_gray_tensor(self.mel_paths[idx])
        elif self.spec_mode == 'cqt':
            spec_t = self._get_gray_tensor(self.cqt_paths[idx])
        else:
            mel_t = self._get_gray_tensor(self.mel_paths[idx])
            cqt_t = self._get_gray_tensor(self.cqt_paths[idx])
            spec_t = torch.cat([mel_t, cqt_t], dim=0)

        acoustic_t = torch.from_numpy(self.axes_7[idx])
        fuzzy_t = torch.from_numpy(self.fuzzy_27[idx])
        label_t = torch.tensor(self.labels[idx], dtype=torch.long)
        weight_t = torch.tensor(self.sample_weights[idx], dtype=torch.float32)
        return spec_t, acoustic_t, fuzzy_t, label_t, weight_t

def create_dataloaders(df_meta: pd.DataFrame, spec_mode: str = 'mel+cqt', batch_size: int = BATCH_SIZE):
    """Tạo bộ DataLoaders có tích hợp WeightedRandomSampler ưu tiên mẫu YouTube cho tập Train."""
    df_train = df_meta[df_meta['split'] == 'train'].copy()
    df_val = df_meta[df_meta['split'] == 'val'].copy()
    df_test_all = df_meta[df_meta['split'] == 'test'].copy()
    df_test_hf = df_meta[(df_meta['split'] == 'test') & (df_meta['data_source'] == 'huggingface')].copy()
    df_test_yt = df_meta[(df_meta['split'] == 'test') & (df_meta['data_source'] == 'youtube')].copy()

    ds_train = UnifiedSpectrogramDataset(df_train, spec_mode=spec_mode, img_size=IMG_SIZE)
    ds_val = UnifiedSpectrogramDataset(df_val, spec_mode=spec_mode, img_size=IMG_SIZE)
    ds_test_all = UnifiedSpectrogramDataset(df_test_all, spec_mode=spec_mode, img_size=IMG_SIZE)
    ds_test_hf = UnifiedSpectrogramDataset(df_test_hf, spec_mode=spec_mode, img_size=IMG_SIZE)
    ds_test_yt = UnifiedSpectrogramDataset(df_test_yt, spec_mode=spec_mode, img_size=IMG_SIZE) if not df_test_yt.empty else None

    # Sử dụng WeightedRandomSampler để tăng tần suất xuất hiện của các mẫu YouTube trong mỗi batch
    train_sampler = WeightedRandomSampler(
        weights=torch.from_numpy(ds_train.sampler_weights),
        num_samples=len(ds_train),
        replacement=True
    )

    pin_mem = torch.cuda.is_available()
    drop_last_train = (len(ds_train) % batch_size == 1)
    loaders = {
        'train': DataLoader(ds_train, batch_size=batch_size, sampler=train_sampler, num_workers=NUM_WORKERS, pin_memory=pin_mem, drop_last=drop_last_train),
        'val': DataLoader(ds_val, batch_size=batch_size, shuffle=False, num_workers=NUM_WORKERS, pin_memory=pin_mem),
        'val_yt': DataLoader(
            UnifiedSpectrogramDataset(df_val[df_val['data_source'] == 'youtube'].copy(), spec_mode=spec_mode, img_size=IMG_SIZE),
            batch_size=batch_size, shuffle=False, num_workers=NUM_WORKERS, pin_memory=pin_mem
        ) if not df_val[df_val['data_source'] == 'youtube'].empty else None,
        'test_all': DataLoader(ds_test_all, batch_size=batch_size, shuffle=False, num_workers=NUM_WORKERS, pin_memory=pin_mem),
        'test_hf': DataLoader(ds_test_hf, batch_size=batch_size, shuffle=False, num_workers=NUM_WORKERS, pin_memory=pin_mem),
        'test_yt': DataLoader(ds_test_yt, batch_size=batch_size, shuffle=False, num_workers=NUM_WORKERS, pin_memory=pin_mem) if ds_test_yt else None,
        'df_test_yt': df_test_yt
    }
    return loaders

print("[OK] Đã khởi tạo thành công lớp UnifiedSpectrogramDataset, Global RAM Cache và SpecAugment!")

---
## PHẦN 3: HỆ THỐNG 21 MÔ HÌNH THỰC NGHIỆM ĐỐI CHỨNG (5 PHƯƠNG ÁN KIẾN TRÚC)
Mỗi phương án kiến trúc triển khai các mô hình đối chứng có kiểm soát chặt chẽ (Ablation Study) nhằm khảo sát từng yếu tố độc lập:

1. **Phương án 1 — Baseline Standard CNN (3 mô hình):**
   - `M1a_Baseline_CNN_Mel_1Ch`: CNN tiêu chuẩn 4 khối trên Đơn phổ Mel (1 kênh).
   - `M1b_Baseline_CNN_CQT_1Ch`: CNN tiêu chuẩn 4 khối trên Đơn phổ CQT (1 kênh).
   - `M1c_Baseline_CNN_MelCQT_2Ch`: CNN tiêu chuẩn 4 khối trên Đa phổ Mel+CQT (2 kênh).
2. **Phương án 2 — CNN + Attention / Transformer Block (5 mô hình):**
   - `M2a`: `CNN + Transformer (Single Mel)` (2L, nhead=4, không SE, đối chứng trực tiếp M1a).
   - `M2b`: `CNN + Transformer (Mel+CQT)` (1L, nhead=4, không SE, gắn 1 lớp Transformer vào đa phổ đối chứng M1c).
   - `M2c`: `CNN + Transformer 2L (Mel+CQT)` (2L, nhead=4, không SE).
   - `M2d`: `CNN + Transformer 4L (Mel+CQT)` (4L, nhead=4, không SE — khảo sát công bằng 1L -> 2L -> 4L).
   - `M2e`: `SE-CNN + Transformer 4L (Mel+CQT)` (SE Channel Attention + 4L, nhead=8).
3. **Phương án 3 — Pretrained Transfer Learning + Transformer Encoder (3 mô hình):**
   - `M3a`: Pretrained **ResNet-34** + Transformer Encoder.
   - `M3b`: Pretrained **EfficientNet-B0** + Transformer Encoder.
   - `M3c`: Pretrained **DenseNet-121** + Transformer Encoder.
4. **Phương án 4 — Multimodal Feature Fusion: Spectrogram + 7 Trục Âm học (5 mô hình):**
   - `M4a`: **CNN+Transformer** + 7 Trục (Concat).
   - `M4b`: **Pretrained ResNet-34** + 7 Trục (Concat).
   - `M4c`: **Pretrained EfficientNet-B0** + 7 Trục (Gated Cross-Modal Attention).
   - `M4d`: **Pretrained EfficientNet-B0** + 7 Trục (Concat — đối chứng 1-1 với M4c).
   - `M4e`: **Pretrained DenseNet-121** + 7 Trục (Concat).
5. **Phương án 5 — Neuro-Symbolic AI tích hợp Fuzzy Rule Loss (5 mô hình, đồng bộ `l_con=0.5, l_rule=0.75`):**
   - `M5a`: Neuro-Symbolic **CNN+Transformer** + Fuzzy Rule Loss.
   - `M5b`: Neuro-Symbolic **Pretrained ResNet-34** + Fuzzy Rule Loss.
   - `M5c`: Neuro-Symbolic **Pretrained EfficientNet-B0 Gated** + Fuzzy Rule Loss.
   - `M5d`: Neuro-Symbolic **Pretrained EfficientNet-B0 Concat** + Fuzzy Rule Loss.
   - `M5e`: Neuro-Symbolic **Pretrained DenseNet-121** + Fuzzy Rule Loss.

In [ ]:
# ==============================================================================
# BƯỚC 3.1: ĐỊNH NGHĨA KIẾN TRÚC PHƯƠNG ÁN 1, PHƯƠNG ÁN 2, PHƯƠNG ÁN 3 VÀ PHƯƠNG ÁN 4
# ==============================================================================
class PositionalEncoding(nn.Module):
    """Mã hóa vị trí hình sin cho chuỗi token không gian-thời gian của Spectrogram."""
    def __init__(self, d_model: int = 128, max_len: int = 512, dropout: float = 0.1):
        super().__init__()
        self.dropout = nn.Dropout(p=dropout)
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0)
        self.register_buffer('pe', pe)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = x + self.pe[:, :x.size(1), :]
        return self.dropout(x)

class SEBlock2D(nn.Module):
    """Khối Squeeze-and-Excitation (Channel Attention) giúp chọn lọc dải tần số quan trọng."""
    def __init__(self, channels: int, reduction: int = 8):
        super().__init__()
        mid_c = max(8, channels // reduction)
        self.se = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
            nn.Linear(channels, mid_c, bias=False),
            nn.ReLU(inplace=True),
            nn.Linear(mid_c, channels, bias=False),
            nn.Sigmoid()
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        w = self.se(x).unsqueeze(-1).unsqueeze(-1)
        return x * w


# ------------------------------------------------------------------------------
# PHƯƠNG ÁN 1: BASELINE STANDARD CNN (M1a: Mel, M1b: CQT, M1c: Mel+CQT)
# ------------------------------------------------------------------------------
class BaselineStandardCNN(nn.Module):
    """
    Phương án 1: Mô hình CNN tiêu chuẩn 4 khối huấn luyện từ đầu (From Scratch)
    làm mốc cơ sở (Baseline Benchmark) cho đơn phổ Mel, đơn phổ CQT và đa phổ Mel+CQT.
    """
    def __init__(self, in_channels: int = 2, num_classes: int = 6):
        super().__init__()
        filters = [32, 64, 128, 256]
        layers_list = []
        curr_in = in_channels
        for f in filters:
            layers_list.append(nn.Sequential(
                nn.Conv2d(curr_in, f, kernel_size=3, padding=1, bias=False),
                nn.BatchNorm2d(f),
                nn.ReLU(inplace=True),
                nn.MaxPool2d(kernel_size=2, stride=2),
                nn.Dropout2d(p=0.2)
            ))
            curr_in = f
        self.conv_blocks = nn.Sequential(*layers_list)
        self.pool = nn.AdaptiveAvgPool2d((4, 4))
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(256 * 4 * 4, 256),
            nn.ReLU(inplace=True),
            nn.Dropout(p=0.4),
            nn.Linear(256, num_classes)
        )

    def forward(self, spec: torch.Tensor, acoustic: Optional[torch.Tensor] = None):
        x = self.conv_blocks(spec)
        x = self.pool(x)
        return self.classifier(x)


# ------------------------------------------------------------------------------
# PHƯƠNG ÁN 2: CNN + ATTENTION / TRANSFORMER BLOCK (M2a, M2b, M2c)
# ------------------------------------------------------------------------------
class CNNTransformerNet(nn.Module):
    """
    Phương án 2: Kết hợp khối tích chập CNN (có tùy chọn SE Channel Attention)
    với Positional Encoding và N tầng Transformer Encoder bắt phụ thuộc toàn cục.
    """
    def __init__(
        self,
        in_channels: int = 2,
        num_classes: int = 6,
        d_model: int = 128,
        nhead: int = 4,
        num_layers: int = 2,
        use_se: bool = False
    ):
        super().__init__()
        b1 = [
            nn.Conv2d(in_channels, 32, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
        ]
        if use_se:
            b1.append(SEBlock2D(32))
        b1.append(nn.MaxPool2d(2, 2))

        b2 = [
            nn.Conv2d(32, 64, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
        ]
        if use_se:
            b2.append(SEBlock2D(64))
        b2.append(nn.MaxPool2d(2, 2))

        b3 = [
            nn.Conv2d(64, d_model, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(d_model),
            nn.ReLU(inplace=True),
        ]
        if use_se:
            b3.append(SEBlock2D(d_model))
        b3.append(nn.AdaptiveAvgPool2d((8, 8)))

        self.cnn_backbone = nn.Sequential(*(b1 + b2 + b3))
        self.pos_encoder = PositionalEncoding(d_model=d_model, max_len=128, dropout=0.1)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=d_model * 2,
            dropout=0.1,
            activation='gelu',
            batch_first=True
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.norm = nn.LayerNorm(d_model)
        self.pool1d = nn.AdaptiveAvgPool1d(1)
        self.classifier = nn.Sequential(
            nn.Linear(d_model, 128),
            nn.GELU(),
            nn.Dropout(0.3),
            nn.Linear(128, num_classes)
        )

    def extract_spec_embedding(self, spec: torch.Tensor) -> torch.Tensor:
        feat = self.cnn_backbone(spec)              # (B, d_model, 8, 8)
        seq = feat.flatten(2).transpose(1, 2)       # (B, 64, d_model)
        seq = self.pos_encoder(seq)
        seq = self.transformer(seq)
        seq = self.norm(seq)
        pooled = self.pool1d(seq.transpose(1, 2)).squeeze(-1)  # (B, d_model)
        return pooled

    def forward(self, spec: torch.Tensor, acoustic: Optional[torch.Tensor] = None):
        z_spec = self.extract_spec_embedding(spec)
        return self.classifier(z_spec)


# ------------------------------------------------------------------------------
# PHƯƠNG ÁN 3: PRETRAINED TRANSFER LEARNING + TRANSFORMER (M3a: ResNet34, M3b: EfficientNetB0, M3c: DenseNet121)
# ------------------------------------------------------------------------------
class PretrainedTransformerNet(nn.Module):
    """
    Phương án 3: Sử dụng Backbone tiền huấn luyện ImageNet (ResNet-34, EfficientNet-B0, DenseNet-121),
    điều chỉnh tầng Conv đầu tiên nhận 2 kênh phổ (Mel+CQT), kết hợp Transformer Encoder Head.
    Cải tiến chiến lược 2-Stage:
      - Stage 1 (Warmup): Mở khóa lớp Conv đầu vào 2 kênh (stem_conv) + Projector + Transformer Head.
      - Stage 2 (Full Fine-tuning): Mở khóa toàn bộ Backbone để thích nghi sâu với miền phổ âm thanh.
    """
    def __init__(self, in_channels: int = 2, num_classes: int = 6, backbone_name: str = 'resnet34', d_model: int = 128):
        super().__init__()
        self.backbone_name = backbone_name.lower()
        if self.backbone_name == 'efficientnet_b0':
            base = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.DEFAULT)
            old_conv = base.features[0][0]
            new_conv = nn.Conv2d(
                in_channels, old_conv.out_channels,
                kernel_size=old_conv.kernel_size, stride=old_conv.stride,
                padding=old_conv.padding, bias=False
            )
            with torch.no_grad():
                mean_w = old_conv.weight.mean(dim=1, keepdim=True)
                new_conv.weight.copy_(mean_w.repeat(1, in_channels, 1, 1))
            base.features[0][0] = new_conv
            self.stem_conv = new_conv
            self.backbone = base.features
            backbone_out_ch = 1280
        elif self.backbone_name == 'densenet121':
            base = models.densenet121(weights=models.DenseNet121_Weights.DEFAULT)
            old_conv = base.features.conv0
            new_conv = nn.Conv2d(
                in_channels, old_conv.out_channels,
                kernel_size=old_conv.kernel_size, stride=old_conv.stride,
                padding=old_conv.padding, bias=False
            )
            with torch.no_grad():
                mean_w = old_conv.weight.mean(dim=1, keepdim=True)
                new_conv.weight.copy_(mean_w.repeat(1, in_channels, 1, 1))
            base.features.conv0 = new_conv
            self.stem_conv = new_conv
            self.backbone = base.features
            backbone_out_ch = 1024
        else:
            base = models.resnet34(weights=models.ResNet34_Weights.DEFAULT)
            old_conv = base.conv1
            new_conv = nn.Conv2d(
                in_channels, old_conv.out_channels,
                kernel_size=old_conv.kernel_size, stride=old_conv.stride,
                padding=old_conv.padding, bias=False
            )
            with torch.no_grad():
                mean_w = old_conv.weight.mean(dim=1, keepdim=True)
                new_conv.weight.copy_(mean_w.repeat(1, in_channels, 1, 1))
            base.conv1 = new_conv
            self.stem_conv = new_conv
            self.backbone = nn.Sequential(
                base.conv1, base.bn1, base.relu, base.maxpool,
                base.layer1, base.layer2, base.layer3, base.layer4
            )
            backbone_out_ch = 512

        self.proj = nn.Sequential(
            nn.Conv2d(backbone_out_ch, d_model, kernel_size=1, bias=False),
            nn.BatchNorm2d(d_model),
            nn.GELU()
        )
        self.pos_encoder = PositionalEncoding(d_model=d_model, max_len=256, dropout=0.1)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=4, dim_feedforward=256,
            dropout=0.1, activation='gelu', batch_first=True
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=2)
        self.norm = nn.LayerNorm(d_model)
        self.pool1d = nn.AdaptiveAvgPool1d(1)
        self.classifier = nn.Sequential(
            nn.Dropout(0.3),
            nn.Linear(d_model, num_classes)
        )
        self.current_stage = 1

    def set_training_stage(self, stage: int = 1):
        """
        Stage 1: Đóng băng các tầng giữa của backbone, nhưng LUÔN MỞ KHÓA tầng stem_conv (2 kênh Mel+CQT)
                 cùng Projector + Transformer Head để mạng học cách tách kênh Mel và CQT ngay từ đầu.
        Stage 2: Mở khóa TOÀN BỘ backbone để tinh chỉnh sâu (Full Domain Adaptation) sang miền ảnh phổ.
        """
        self.current_stage = stage
        if stage == 1:
            for p in self.backbone.parameters():
                p.requires_grad = False
            for p in self.stem_conv.parameters():
                p.requires_grad = True
        else:
            for p in self.backbone.parameters():
                p.requires_grad = True

    def train(self, mode: bool = True):
        super().train(mode)
        # Trong Stage 1, giữ các lớp BatchNorm2d đang đóng băng của backbone ở chế độ eval()
        # để bảo toàn thống kê running_mean / running_var của ImageNet trước khi sang Stage 2.
        if mode and self.current_stage == 1:
            for m in self.backbone.modules():
                if isinstance(m, (nn.BatchNorm2d, nn.SyncBatchNorm)):
                    m.eval()
        return self

    def extract_spec_embedding(self, spec: torch.Tensor) -> torch.Tensor:
        feat = self.backbone(spec)
        if self.backbone_name == 'densenet121':
            feat = F.relu(feat, inplace=True)
        feat = self.proj(feat)                      # (B, 128, H', W')
        seq = feat.flatten(2).transpose(1, 2)       # (B, H'*W', 128)
        seq = self.pos_encoder(seq)
        seq = self.transformer(seq)
        seq = self.norm(seq)
        z_spec = self.pool1d(seq.transpose(1, 2)).squeeze(-1)  # (B, 128)
        return z_spec

    def forward(self, spec: torch.Tensor, acoustic: Optional[torch.Tensor] = None):
        z_spec = self.extract_spec_embedding(spec)
        return self.classifier(z_spec)


# ------------------------------------------------------------------------------
# PHƯƠNG ÁN 4: MULTIMODAL FEATURE FUSION (M4a: CNNTransformer Concat, M4b: ResNet34 Concat, M4c: EfficientNetB0 Gated)
# ------------------------------------------------------------------------------
class MultimodalFusionNet(nn.Module):
    """
    Phương án 4: Hợp nhất đa phương thức giữa Spectrogram (z_spec 128 chiều) và 7 Trục Âm học (z_acoustic 32 chiều):
      - Hỗ trợ spec_backbone: 'cnn_transformer', 'resnet34', 'efficientnet_b0', 'densenet121'.
      - Hỗ trợ fusion_mode: 'concat' (ghép nối trực tiếp) hoặc 'gated' (Gated Cross-Modal Attention).
    """
    def __init__(
        self,
        in_channels: int = 2,
        num_classes: int = 6,
        acoustic_dim: int = 7,
        spec_backbone: str = 'cnn_transformer',
        fusion_mode: str = 'concat'
    ):
        super().__init__()
        self.spec_backbone_type = spec_backbone.lower()
        self.fusion_mode = fusion_mode.lower()

        if self.spec_backbone_type == 'cnn_transformer':
            self.spec_branch = CNNTransformerNet(
                in_channels=in_channels, num_classes=num_classes, d_model=128, nhead=4, num_layers=3, use_se=True
            )
        else:
            self.spec_branch = PretrainedTransformerNet(
                in_channels=in_channels, num_classes=num_classes, backbone_name=self.spec_backbone_type, d_model=128
            )

        self.acoustic_mlp = nn.Sequential(
            nn.Linear(acoustic_dim, 64),
            nn.LayerNorm(64),
            nn.ReLU(inplace=True),
            nn.Dropout(0.2),
            nn.Linear(64, 32),
            nn.LayerNorm(32),
            nn.ReLU(inplace=True)
        )

        if self.fusion_mode == 'gated':
            # Cổng điều tiết Cross-Modal Attention học trọng số đóng góp của từng chiều đặc trưng
            self.gate_layer = nn.Sequential(
                nn.Linear(128 + 32, 160),
                nn.Sigmoid()
            )

        self.fusion_classifier = nn.Sequential(
            nn.Linear(128 + 32, 128),
            nn.LayerNorm(128),
            nn.GELU(),
            nn.Dropout(0.3),
            nn.Linear(128, num_classes)
        )

    def set_training_stage(self, stage: int = 1):
        if hasattr(self.spec_branch, 'set_training_stage'):
            self.spec_branch.set_training_stage(stage)

    def extract_fused_embedding(self, spec: torch.Tensor, acoustic: torch.Tensor) -> torch.Tensor:
        z_spec = self.spec_branch.extract_spec_embedding(spec)      # (B, 128)
        z_acoustic = self.acoustic_mlp(acoustic)                    # (B, 32)
        z_concat = torch.cat([z_spec, z_acoustic], dim=1)           # (B, 160)
        if self.fusion_mode == 'gated':
            gate = self.gate_layer(z_concat)
            return z_concat * gate
        return z_concat

    def forward(self, spec: torch.Tensor, acoustic: torch.Tensor):
        z_fused = self.extract_fused_embedding(spec, acoustic)
        return self.fusion_classifier(z_fused)

print("[OK] Đã định nghĩa thành công toàn bộ kiến trúc Phương án 1, 2, 3 và 4!")

In [ ]:
# ==============================================================================
# BƯỚC 3.2: BỘ SUY LUẬN LUẬT MỜ KHẢ VI (DIFFERENTIABLE FUZZY RULE LOSS) & PHƯƠNG ÁN 5
# ==============================================================================
RULE_VAR_TO_COL_IDX = {
    'Energy_Rất_yếu': 0, 'Energy_Very_Weak': 0,
    'Energy_Yếu': 1, 'Energy_Weak': 1,
    'Energy_Vừa': 2, 'Energy_Moderate': 2,
    'Energy_Mạnh': 3, 'Energy_Strong': 3,
    'Energy_Rất_mạnh': 4, 'Energy_Very_Strong': 4,
    'Harmony_Thấp': 5, 'Harmony_Low': 5,
    'Harmony_Trung_bình': 6, 'Harmony_Medium': 6,
    'Harmony_Cao': 7, 'Harmony_High': 7,
    'Pitch_Rất_trầm': 8, 'Pitch_Very_Low': 8,
    'Pitch_Trầm': 9, 'Pitch_Low': 9,
    'Pitch_Trung': 10, 'Pitch_Medium': 10,
    'Pitch_Cao': 11, 'Pitch_High': 11,
    'Pitch_Rất_cao': 12, 'Pitch_Very_High': 12,
    'Spectral_Thấp': 13, 'Spectral_Low': 13,
    'Spectral_Trung_bình': 14, 'Spectral_Medium': 14,
    'Spectral_Cao': 15, 'Spectral_High': 15,
    'Tempo_Chậm': 16, 'Tempo_Slow': 16,
    'Tempo_Vừa': 17, 'Tempo_Moderate': 17,
    'Tempo_Nhanh': 18, 'Tempo_Fast': 18,
    'Timbre_Rất_thấp': 19, 'Timbre_Very_Low': 19,
    'Timbre_Thấp': 20, 'Timbre_Low': 20,
    'Timbre_Trung_bình': 21, 'Timbre_Medium': 21,
    'Timbre_Cao': 22, 'Timbre_High': 22,
    'Timbre_Rất_cao': 23, 'Timbre_Very_High': 23,
    'Vibrato_Ít_luyến': 24, 'Vibrato_Light_Ornament': 24,
    'Vibrato_Vừa': 25, 'Vibrato_Moderate': 25,
    'Vibrato_Nhiều_luyến': 26, 'Vibrato_Heavy_Ornament': 26,
}

# Danh sách 30 Luật mờ IF-THEN chuẩn trích xuất từ rules_decision_tree_continuous.json & rules_random_forest_top.json
# Được nhúng trực tiếp làm phương án dự phòng an toàn tuyệt đối nếu trên Google Drive quên tải file .json lên
EMBEDDED_CONTINUOUS_FUZZY_RULES = [
    # --- Top Rules từ Random Forest Continuous (Confidence cao 70% - 96%) ---
    {"rule_text": "Energy_Rất_mạnh <= 0.294 AND Pitch_Trầm > 0.300 AND Energy_Rất_yếu > 0.479 AND Timbre_Rất_cao <= 0.325 AND Pitch_Rất_trầm <= 0.497", "consequent": "Ca trù", "confidence": 0.7664},
    {"rule_text": "Energy_Rất_yếu > 0.082 AND Timbre_Thấp > 0.178 AND Energy_Rất_yếu > 0.398 AND Harmony_Trung_bình > 0.333 AND Harmony_Cao <= 0.333", "consequent": "Ca trù", "confidence": 0.8584},
    {"rule_text": "Timbre_Rất_cao <= 0.182 AND Harmony_Thấp <= 0.333 AND Pitch_Cao > 0.019 AND Pitch_Rất_cao <= 0.026 AND Harmony_Cao > 0.334", "consequent": "Chèo", "confidence": 0.8941},
    {"rule_text": "Timbre_Rất_cao <= 0.184 AND Harmony_Cao > 0.333 AND Pitch_Rất_cao > 0.005 AND Energy_Rất_yếu > 0.053 AND Vibrato_Nhiều_luyến > 0.044", "consequent": "Chèo", "confidence": 0.9067},
    {"rule_text": "Harmony_Cao > 0.333 AND Spectral_Trung_bình <= 0.220 AND Timbre_Rất_thấp <= 0.197 AND Pitch_Trầm > 0.365 AND Energy_Rất_mạnh <= 0.140", "consequent": "Cải lương", "confidence": 0.9073},
    {"rule_text": "Spectral_Cao <= 0.168 AND Harmony_Cao > 0.333 AND Pitch_Trầm > 0.298 AND Vibrato_Ít_luyến <= 0.880 AND Energy_Yếu > 0.369", "consequent": "Cải lương", "confidence": 0.8698},
    {"rule_text": "Energy_Rất_yếu <= 0.129 AND Spectral_Trung_bình > 0.184 AND Pitch_Trầm <= 0.337 AND Harmony_Cao > 0.333 AND Energy_Mạnh <= 0.237", "consequent": "Chầu văn", "confidence": 0.8224},
    {"rule_text": "Energy_Vừa > 0.145 AND Pitch_Trầm <= 0.371 AND Timbre_Rất_thấp <= 0.165 AND Energy_Rất_yếu <= 0.123 AND Spectral_Trung_bình > 0.244", "consequent": "Chầu văn", "confidence": 0.8442},
    {"rule_text": "Harmony_Cao <= 0.333 AND Energy_Rất_yếu <= 0.318 AND Timbre_Rất_thấp <= 0.184 AND Tempo_Vừa > 0.498 AND Pitch_Trung > 0.036", "consequent": "Quan họ", "confidence": 0.9222},
    {"rule_text": "Harmony_Thấp > 0.333 AND Energy_Rất_yếu <= 0.289 AND Timbre_Rất_thấp <= 0.194 AND Tempo_Vừa > 0.499 AND Pitch_Rất_trầm <= 0.409", "consequent": "Quan họ", "confidence": 0.9163},
    {"rule_text": "Timbre_Rất_thấp > 0.191 AND Vibrato_Nhiều_luyến <= 0.135 AND Pitch_Rất_trầm <= 0.464 AND Energy_Yếu <= 0.320 AND Energy_Rất_mạnh <= 0.251", "consequent": "Hát xẩm", "confidence": 0.9469},
    {"rule_text": "Timbre_Rất_thấp > 0.192 AND Pitch_Rất_trầm <= 0.453 AND Vibrato_Ít_luyến > 0.562 AND Energy_Mạnh > 0.136 AND Harmony_Trung_bình <= 0.333", "consequent": "Hát xẩm", "confidence": 0.9616},
    # --- Các nhánh luật phân hoạch từ Decision Tree Continuous ---
    {"rule_text": "Timbre_Rất_thấp <= 0.191 AND Harmony_Cao <= 0.333 AND Tempo_Vừa <= 0.497 AND Pitch_Trung <= 0.103 AND Spectral_Trung_bình > 0.236", "consequent": "Chầu văn", "confidence": 0.6667},
    {"rule_text": "Timbre_Rất_thấp <= 0.191 AND Harmony_Cao <= 0.333 AND Tempo_Vừa <= 0.497 AND Pitch_Trung > 0.103 AND Spectral_Thấp > 0.733", "consequent": "Ca trù", "confidence": 0.5556},
    {"rule_text": "Timbre_Rất_thấp <= 0.191 AND Harmony_Cao <= 0.333 AND Tempo_Vừa > 0.497 AND Energy_Rất_yếu <= 0.332 AND Pitch_Trung > 0.035", "consequent": "Quan họ", "confidence": 0.872},
    {"rule_text": "Timbre_Rất_thấp <= 0.191 AND Harmony_Cao <= 0.333 AND Tempo_Vừa > 0.497 AND Energy_Rất_yếu > 0.332 AND Spectral_Thấp <= 0.468", "consequent": "Cải lương", "confidence": 0.7273},
    {"rule_text": "Timbre_Rất_thấp <= 0.191 AND Harmony_Cao <= 0.333 AND Tempo_Vừa > 0.497 AND Energy_Rất_yếu > 0.332 AND Spectral_Thấp > 0.468", "consequent": "Ca trù", "confidence": 0.70},
    {"rule_text": "Timbre_Rất_thấp <= 0.191 AND Harmony_Cao > 0.333 AND Energy_Vừa <= 0.168 AND Energy_Yếu > 0.429 AND Pitch_Trầm <= 0.355", "consequent": "Chầu văn", "confidence": 0.6744},
    {"rule_text": "Timbre_Rất_thấp <= 0.191 AND Harmony_Cao > 0.333 AND Energy_Vừa <= 0.168 AND Energy_Yếu > 0.429 AND Pitch_Trầm > 0.355", "consequent": "Cải lương", "confidence": 0.7931},
    {"rule_text": "Timbre_Rất_thấp <= 0.191 AND Harmony_Cao > 0.333 AND Energy_Vừa > 0.168 AND Pitch_Trầm <= 0.380 AND Spectral_Trung_bình > 0.219", "consequent": "Chầu văn", "confidence": 0.7295},
    {"rule_text": "Timbre_Rất_thấp <= 0.191 AND Harmony_Cao > 0.333 AND Energy_Vừa > 0.168 AND Pitch_Trầm > 0.380 AND Spectral_Trung_bình <= 0.235", "consequent": "Cải lương", "confidence": 0.8588},
    {"rule_text": "Timbre_Rất_thấp > 0.191 AND Vibrato_Nhiều_luyến <= 0.157 AND Pitch_Rất_trầm <= 0.453 AND Energy_Rất_yếu <= 0.194 AND Tempo_Nhanh <= 0.468", "consequent": "Hát xẩm", "confidence": 0.9036},
    {"rule_text": "Timbre_Rất_thấp > 0.191 AND Vibrato_Nhiều_luyến <= 0.157 AND Pitch_Rất_trầm <= 0.453 AND Energy_Rất_yếu > 0.194 AND Harmony_Thấp > 0.333", "consequent": "Ca trù", "confidence": 0.7778},
    {"rule_text": "Timbre_Rất_thấp > 0.191 AND Vibrato_Nhiều_luyến > 0.157 AND Energy_Rất_yếu > 0.258 AND Harmony_Cao <= 0.333 AND Spectral_Thấp <= 0.562", "consequent": "Ca trù", "confidence": 0.8229},
]

class DifferentiableFuzzyRuleLoss(nn.Module):
    """
    Biên dịch tập luật IF-THEN từ data/rules/rules_decision_tree_continuous.json
    và data/rules/rules_random_forest_top.json (hoặc dùng EMBEDDED_CONTINUOUS_FUZZY_RULES nếu thiếu file)
    thành các phép toán Tensor khả vi (Differentiable T-norm & ReLU Implication Penalty).
    """
    def __init__(self, rules_json_path: Path, temperature: float = 12.0):
        super().__init__()
        self.temperature = temperature
        self.parsed_rules = []

        dt_candidates = [
            rules_json_path,
            PROJECT_ROOT / 'data' / 'rules' / 'rules_decision_tree_continuous.json',
            PROJECT_ROOT / 'rules_decision_tree_continuous.json',
            DIRS['rules'] / 'rules_decision_tree_continuous.json',
            Path('data/rules/rules_decision_tree_continuous.json'),
            Path('/content/drive/MyDrive/Colab Notebooks/LuanVan/data/rules/rules_decision_tree_continuous.json')
        ]
        rf_candidates = [
            PROJECT_ROOT / 'data' / 'rules' / 'rules_random_forest_top.json',
            DIRS['rules'] / 'rules_random_forest_top.json',
            Path('data/rules/rules_random_forest_top.json'),
            Path('/content/drive/MyDrive/Colab Notebooks/LuanVan/data/rules/rules_random_forest_top.json')
        ]

        raw_rules_list = []
        loaded_sources = []

        resolved_dt = next((p for p in dt_candidates if p and Path(p).exists()), None)
        if resolved_dt is not None:
            with open(resolved_dt, 'r', encoding='utf-8') as f:
                raw_rules_list.extend(json.load(f))
            loaded_sources.append(resolved_dt.name)

        resolved_rf = next((p for p in rf_candidates if p and Path(p).exists()), None)
        if resolved_rf is not None:
            with open(resolved_rf, 'r', encoding='utf-8') as f:
                rf_dict = json.load(f)
                if isinstance(rf_dict, dict):
                    for g_rules in rf_dict.values():
                        raw_rules_list.extend(g_rules)
                elif isinstance(rf_dict, list):
                    raw_rules_list.extend(rf_dict)
            loaded_sources.append(resolved_rf.name)

        if not raw_rules_list:
            raw_rules_list = EMBEDDED_CONTINUOUS_FUZZY_RULES
            loaded_sources.append("EMBEDDED_CONTINUOUS_FUZZY_RULES (Fallback an toàn)")

        seen_rule_texts = set()
        for r in raw_rules_list:
            consequent = r.get('consequent', '')
            if consequent not in GENRE_TO_IDX:
                continue
            target_idx = GENRE_TO_IDX[consequent]
            conf = float(r.get('confidence', 0.5))
            rule_text = r.get('rule_text', '').strip()
            if not rule_text or rule_text in seen_rule_texts:
                continue
            seen_rule_texts.add(rule_text)

            clauses = []
            for part in rule_text.split('AND'):
                part = part.strip()
                m = re.match(r'^([A-Za-zÀ-ỹ_]+)\s*(<=|>)\s*([0-9.]+)$', part)
                if m:
                    var_name, op, th_str = m.group(1), m.group(2), float(m.group(3))
                    if var_name in RULE_VAR_TO_COL_IDX:
                        col_idx = RULE_VAR_TO_COL_IDX[var_name]
                        clauses.append((col_idx, 1.0 if op == '>' else -1.0, th_str))
            if clauses:
                self.parsed_rules.append({
                    'clauses': clauses,
                    'consequent_idx': target_idx,
                    'confidence': conf,
                    'rule_text': rule_text,
                    'consequent_name': consequent
                })
        print(f"[Neuro-Symbolic] Đã biên dịch {len(self.parsed_rules)} luật mờ IF-THEN từ: {', '.join(loaded_sources)}")

    def compute_rule_activations(self, concept_probs: torch.Tensor) -> List[Tuple[torch.Tensor, int, float]]:
        """Tính độ kích hoạt (firing strength) [0, 1] của từng luật trên batch dữ liệu."""
        activations = []
        for r in self.parsed_rules:
            clause_probs = []
            for col_idx, sign, th in r['clauses']:
                val = concept_probs[:, col_idx]
                deg = torch.sigmoid(sign * (val - th) * self.temperature)
                clause_probs.append(deg)
            firing = torch.stack(clause_probs, dim=-1).amin(dim=-1)
            activations.append((firing, r['consequent_idx'], r['confidence']))
        return activations

    def forward(self, genre_logits: torch.Tensor, concept_probs: torch.Tensor) -> torch.Tensor:
        if not self.parsed_rules:
            return torch.tensor(0.0, device=genre_logits.device)
        genre_probs = torch.softmax(genre_logits, dim=-1)
        rule_acts = self.compute_rule_activations(concept_probs)

        penalties = []
        for firing, target_idx, conf in rule_acts:
            p_target = genre_probs[:, target_idx]
            violation = F.relu(firing - p_target)
            penalties.append(conf * firing * (violation ** 2))

        return torch.stack(penalties, dim=-1).mean()

    @torch.no_grad()
    def compute_violation_rate(self, genre_logits: torch.Tensor, concept_probs: torch.Tensor, firing_th: float = 0.35) -> float:
        """
        Tính tỷ lệ mẫu vi phạm hệ luật (%): Một mẫu bị coi là vi phạm luật nếu
        thể loại mô hình dự đoán (argmax) mâu thuẫn với thể loại kết luận của luật mờ
        có độ kích hoạt (firing * confidence) cao nhất đối với đặc trưng của mẫu đó.
        """
        if not self.parsed_rules:
            return 0.0
        preds = torch.argmax(genre_logits, dim=-1)
        B = genre_logits.size(0)
        rule_acts = self.compute_rule_activations(concept_probs)

        scores_per_rule = torch.stack([firing * conf for firing, _, conf in rule_acts], dim=-1)
        raw_firings = torch.stack([firing for firing, _, _ in rule_acts], dim=-1)
        consequents = torch.tensor([c_idx for _, c_idx, _ in rule_acts], device=genre_logits.device)

        best_r_idx = torch.argmax(scores_per_rule, dim=-1)
        max_firing = raw_firings[torch.arange(B), best_r_idx]
        expected_genre = consequents[best_r_idx]

        active_mask = max_firing >= firing_th
        if active_mask.sum().item() > 0:
            violated = (preds[active_mask] != expected_genre[active_mask]).float().mean().item()
        else:
            violated = (preds != expected_genre).float().mean().item()
        return round(violated * 100.0, 2)


# ------------------------------------------------------------------------------
# PHƯƠNG ÁN 5: NEURO-SYMBOLIC MODEL (M5a: CNNTransformer, M5b: ResNet34, M5c: EfficientNetB0 Gated)
# ------------------------------------------------------------------------------
class NeuroSymbolicFuzzyNet(nn.Module):
    """
    Phương án 5: Mô hình Tượng trưng - Nơ-ron (Neuro-Symbolic AI) kế thừa kiến trúc
    Multimodal (Phương án 4), xuất đồng thời:
      1. Logits dự đoán 6 thể loại nhạc cổ truyền.
      2. Vector xác suất kích hoạt 27 khái niệm âm học mờ (Concept Bottleneck Head).
    Hợp nhất thông tin từ Concept Head vào Classifier và huấn luyện với Fuzzy Rule Loss.
    """
    def __init__(
        self,
        in_channels: int = 2,
        num_classes: int = 6,
        acoustic_dim: int = 7,
        num_concepts: int = 27,
        spec_backbone: str = 'cnn_transformer',
        fusion_mode: str = 'concat'
    ):
        super().__init__()
        self.base_multimodal = MultimodalFusionNet(
            in_channels=in_channels,
            num_classes=num_classes,
            acoustic_dim=acoustic_dim,
            spec_backbone=spec_backbone,
            fusion_mode=fusion_mode
        )
        self.concept_head = nn.Sequential(
            nn.Linear(160, 64),
            nn.LayerNorm(64),
            nn.GELU(),
            nn.Linear(64, num_concepts),
            nn.Sigmoid()
        )
        self.ns_classifier = nn.Sequential(
            nn.Linear(160 + num_concepts, 128),
            nn.LayerNorm(128),
            nn.GELU(),
            nn.Dropout(0.3),
            nn.Linear(128, num_classes)
        )

    def set_training_stage(self, stage: int = 1):
        self.base_multimodal.set_training_stage(stage)

    def forward(self, spec: torch.Tensor, acoustic: torch.Tensor, return_concepts: bool = False):
        z_fused = self.base_multimodal.extract_fused_embedding(spec, acoustic)  # (B, 160)
        concept_probs = self.concept_head(z_fused)                              # (B, 27)
        combined = torch.cat([z_fused, concept_probs], dim=1)                   # (B, 187)
        genre_logits = self.ns_classifier(combined)                             # (B, 6)
        if return_concepts:
            return genre_logits, concept_probs
        return genre_logits

rules_file_path = DIRS['rules'] / 'rules_decision_tree_continuous.json'
fuzzy_rule_criterion = DifferentiableFuzzyRuleLoss(rules_file_path).to(DEVICE)

---
## PHẦN 4: ENGINE HUẤN LUYỆN 50 EPOCHS, PHẠT NẶNG MẪU YOUTUBE, MIXED PRECISION (AMP) & ĐÁNH GIÁ TOÀN DIỆN
- **Thời lượng huấn luyện nâng cấp:** Tối đa **50 Epochs** cho mỗi mô hình (Stage 1: 10 Epochs Warmup với `lr=1e-3`; Stage 2: 40 Epochs Full Fine-tuning với `lr=2e-4`), kết hợp Early Stopping thông minh (`patience=12` sau tối thiểu 28 epochs) giúp mô hình hội tụ sâu trên cả phổ Mel và CQT.
- **Phạt nặng mẫu YouTube (Cost-Sensitive Domain Penalty):** Các mẫu YouTube bị đoán sai chịu mức phạt gấp **3.5x đến 5.9x** (ưu tiên các thể loại ít mẫu như *Ca trù YouTube*) kết hợp cùng `WeightedRandomSampler` và chọn Checkpoint theo điểm hợp thành `0.55 * YT_Val_F1 + 0.45 * Val_F1`.
- **Mixed Precision:** `torch.amp.autocast('cuda')` và `GradScaler('cuda')` giúp tăng tốc độ huấn luyện trên GPU T4 và tiết kiệm VRAM.
- **Lưu trữ tự động:** Lưu trọng số tốt nhất vào `experiments/checkpoints/{model_key}_best.pth` và lịch sử vào `{model_key}_history.json`.

In [ ]:
# ==============================================================================
# BƯỚC 4.1: ENGINE HUẤN LUYỆN & ĐÁNH GIÁ CHUNG CHO TOÀN BỘ 15 MÔ HÌNH
# ==============================================================================
def count_parameters(model: nn.Module) -> Tuple[int, int]:
    total_p = sum(p.numel() for p in model.parameters())
    train_p = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return total_p, train_p

@torch.no_grad()
def evaluate_model(
    model: nn.Module,
    loader: Optional[DataLoader],
    rule_engine: DifferentiableFuzzyRuleLoss,
    is_neuro_symbolic: bool = False
) -> Dict[str, Any]:
    """Đánh giá toàn diện mô hình trên một DataLoader."""
    if loader is None or len(loader.dataset) == 0:
        return {
            'loss': 0.0, 'acc': 0.0, 'bal_acc': 0.0,
            'macro_prec': 0.0, 'macro_rec': 0.0, 'macro_f1': 0.0,
            'rule_violation_rate': 0.0, 'y_true': [], 'y_pred': [], 'y_probs': np.zeros((0, 6))
        }

    model.eval()
    ce_loss_fn = nn.CrossEntropyLoss()
    total_loss = 0.0
    all_preds, all_targets, all_probs = [], [], []
    all_logits_t, all_concepts_t = [], []

    for batch in loader:
        spec, acoustic, fuzzy_gt, labels = batch[0], batch[1], batch[2], batch[3]
        spec = spec.to(DEVICE, non_blocking=True)
        acoustic = acoustic.to(DEVICE, non_blocking=True)
        fuzzy_gt = fuzzy_gt.to(DEVICE, non_blocking=True)
        labels = labels.to(DEVICE, non_blocking=True)

        if is_neuro_symbolic:
            logits, _ = model(spec, acoustic, return_concepts=True)
            loss = ce_loss_fn(logits, labels)
        else:
            logits = model(spec, acoustic)
            loss = ce_loss_fn(logits, labels)
        # Luôn đánh giá tỷ lệ vi phạm luật trên cùng chuẩn đối chiếu Ground-Truth Fuzzy Concepts (fuzzy_gt)
        # để đảm bảo công bằng tuyệt đối khi so sánh toàn bộ 15 mô hình trên cùng thước đo
        all_concepts_t.append(fuzzy_gt.detach())

        total_loss += loss.item() * labels.size(0)
        probs = torch.softmax(logits, dim=-1)
        preds = torch.argmax(probs, dim=-1)

        all_logits_t.append(logits.detach())
        all_probs.append(probs.cpu().numpy())
        all_preds.extend(preds.cpu().tolist())
        all_targets.extend(labels.cpu().tolist())

    n_samples = len(all_targets)
    avg_loss = total_loss / max(1, n_samples)
    acc = accuracy_score(all_targets, all_preds) * 100.0
    bal_acc = balanced_accuracy_score(all_targets, all_preds) * 100.0
    macro_prec = precision_score(all_targets, all_preds, average='macro', zero_division=0) * 100.0
    macro_rec = recall_score(all_targets, all_preds, average='macro', zero_division=0) * 100.0
    macro_f1 = f1_score(all_targets, all_preds, average='macro', zero_division=0) * 100.0

    cat_logits = torch.cat(all_logits_t, dim=0)
    cat_concepts = torch.cat(all_concepts_t, dim=0)
    rv_rate = rule_engine.compute_violation_rate(cat_logits, cat_concepts)

    return {
        'loss': round(avg_loss, 4),
        'acc': round(acc, 2),
        'bal_acc': round(bal_acc, 2),
        'macro_prec': round(macro_prec, 2),
        'macro_rec': round(macro_rec, 2),
        'macro_f1': round(macro_f1, 2),
        'rule_violation_rate': round(rv_rate, 2),
        'y_true': all_targets,
        'y_pred': all_preds,
        'y_probs': np.concatenate(all_probs, axis=0)
    }

def train_experiment_model(
    model_key: str,
    model: nn.Module,
    loaders: Dict[str, Any],
    rule_engine: DifferentiableFuzzyRuleLoss,
    epochs_stage1: int = 50,
    epochs_stage2: int = 0,
    lr_stage1: float = 1e-3,
    lr_stage2: float = 2e-4,
    is_neuro_symbolic: bool = False,
    lambda_concept: float = 0.4,
    lambda_rule: float = 0.5,
    force_retrain: bool = False,
    legacy_key: Optional[str] = None
) -> Dict[str, Any]:
    """
    Huấn luyện mô hình (tối đa 50 epochs) với Mixed Precision (AMP), ReduceLROnPlateau,
    Early Stopping thông minh và tự động lưu checkpoint tốt nhất về Google Drive.
    """
    ckpt_path = DIRS['checkpoints'] / f"{model_key}_best.pth"
    hist_path = DIRS['checkpoints'] / f"{model_key}_history.json"

    if (not ckpt_path.exists() or not hist_path.exists()) and legacy_key and not force_retrain:
        leg_ckpt = DIRS['checkpoints'] / f"{legacy_key}_best.pth"
        leg_hist = DIRS['checkpoints'] / f"{legacy_key}_history.json"
        if leg_ckpt.exists() and leg_hist.exists():
            shutil.copy2(leg_ckpt, ckpt_path)
            shutil.copy2(leg_hist, hist_path)
            print(f"\n[MIGRATE] Tự động kế thừa Checkpoint từ {legacy_key} sang {model_key}")

    if ckpt_path.exists() and hist_path.exists() and not force_retrain:
        print(f"\n[RESUME] Phát hiện Checkpoint đã huấn luyện của {model_key} tại: {ckpt_path.name}")
        ckpt = torch.load(ckpt_path, map_location=DEVICE, weights_only=False)
        model.load_state_dict(ckpt['model_state_dict'])
        with open(hist_path, 'r', encoding='utf-8') as f:
            history = json.load(f)
        return history

    print("\n" + "=" * 88)
    print(f"BẮT ĐẦU HUẤN LUYỆN MÔ HÌNH: {model_key}")
    total_p, train_p = count_parameters(model)
    print(f"Tổng tham số: {total_p:,} | Tham số huấn luyện: {train_p:,}")
    print("=" * 88)

    spec_aug = SpecAugment(freq_mask_param=14, time_mask_param=20, p=0.5).to(DEVICE)
    # Sử dụng reduction='none' để nhân trực tiếp hệ số phạt nặng (sample_weight) cho từng mẫu YouTube
    ce_criterion_none = nn.CrossEntropyLoss(label_smoothing=0.05, reduction='none')
    use_amp = torch.cuda.is_available()
    scaler = torch.amp.GradScaler('cuda', enabled=use_amp)

    history = {
        'train_loss': [], 'val_loss': [],
        'train_acc': [], 'val_acc': [], 'val_f1': [], 'val_yt_f1': [],
        'val_rule_violation': [], 'epoch_times': []
    }
    best_val_score = -1.0
    best_val_f1 = -1.0
    best_weights = None

    stages = []
    if epochs_stage2 > 0 and hasattr(model, 'set_training_stage'):
        stages.append((1, epochs_stage1, lr_stage1))
        stages.append((2, epochs_stage2, lr_stage2))
    else:
        if hasattr(model, 'set_training_stage'):
            model.set_training_stage(2)
        stages.append((1, epochs_stage1 + epochs_stage2, lr_stage1))

    global_epoch = 0
    for stage_idx, n_epochs, lr in stages:
        if hasattr(model, 'set_training_stage') and len(stages) > 1:
            model.set_training_stage(stage_idx)
            _, tr_p = count_parameters(model)
            print(f"\n>>> Chuyển sang Stage {stage_idx} (lr={lr:.1e}, trainable_params={tr_p:,}) <<<")

        trainable_params = [p for p in model.parameters() if p.requires_grad]
        optimizer = torch.optim.AdamW(trainable_params, lr=lr, weight_decay=1e-4)
        scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
            optimizer, mode='max', factor=0.5, patience=4
        )
        patience_counter = 0
        max_patience = 12

        for ep in range(1, n_epochs + 1):
            global_epoch += 1
            t_start = time.time()
            model.train()
            running_loss, correct, total = 0.0, 0, 0

            for spec, acoustic, fuzzy_gt, labels, sample_w in loaders['train']:
                spec = spec.to(DEVICE, non_blocking=True)
                acoustic = acoustic.to(DEVICE, non_blocking=True)
                fuzzy_gt = fuzzy_gt.to(DEVICE, non_blocking=True)
                labels = labels.to(DEVICE, non_blocking=True)
                sample_w = sample_w.to(DEVICE, non_blocking=True)

                spec_aug_in = spec_aug(spec)
                optimizer.zero_grad(set_to_none=True)

                amp_device = 'cuda' if torch.cuda.is_available() else 'cpu'
                with torch.amp.autocast(amp_device, enabled=use_amp):
                    if is_neuro_symbolic:
                        logits, concept_pred = model(spec_aug_in, acoustic, return_concepts=True)
                        per_sample_ce = ce_criterion_none(logits, labels)
                        # Phạt nặng các mẫu YouTube bị đoán sai theo trọng số sample_w (3.5x - 6.0x)
                        loss_ce = (per_sample_ce * sample_w).mean()
                        # Kết hợp phạt vi phạm luật trên cả dự đoán Concept Head và Ground-Truth Fuzzy Concepts
                        loss_rule = 0.5 * rule_engine(logits, concept_pred) + 0.5 * rule_engine(logits, fuzzy_gt)
                    else:
                        logits = model(spec_aug_in, acoustic)
                        per_sample_ce = ce_criterion_none(logits, labels)
                        # Phạt nặng các mẫu YouTube bị đoán sai theo trọng số sample_w (3.5x - 6.0x)
                        loss_ce = (per_sample_ce * sample_w).mean()

                if is_neuro_symbolic:
                    with torch.amp.autocast(amp_device, enabled=False):
                        per_sample_con = F.binary_cross_entropy(
                            concept_pred.float().clamp(1e-6, 1.0 - 1e-6), fuzzy_gt.float(), reduction='none'
                        ).mean(dim=-1)
                        loss_con = (per_sample_con * sample_w.float()).mean()
                    loss = loss_ce + lambda_concept * loss_con + lambda_rule * loss_rule
                else:
                    loss = loss_ce

                scaler.scale(loss).backward()
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(trainable_params, max_norm=2.0)
                scaler.step(optimizer)
                scaler.update()

                running_loss += loss.item() * labels.size(0)
                preds = torch.argmax(logits, dim=-1)
                correct += (preds == labels).sum().item()
                total += labels.size(0)

            ep_time = time.time() - t_start
            train_loss = running_loss / max(1, total)
            train_acc = (correct / max(1, total)) * 100.0

            val_metrics = evaluate_model(model, loaders['val'], rule_engine, is_neuro_symbolic=is_neuro_symbolic)
            val_loss = val_metrics['loss']
            val_acc = val_metrics['acc']
            val_f1 = val_metrics['macro_f1']
            val_rv = val_metrics['rule_violation_rate']

            # Đánh giá riêng trên tập Validation YouTube để chọn Checkpoint có khả năng khái quát hóa thực tế tốt nhất
            if loaders.get('val_yt') is not None:
                val_yt_metrics = evaluate_model(model, loaders['val_yt'], rule_engine, is_neuro_symbolic=is_neuro_symbolic)
                val_yt_f1 = val_yt_metrics['macro_f1']
                # Điểm chọn Checkpoint ưu tiên 55% hiệu năng trên YouTube Validation + 45% hiệu năng tổng hợp
                val_selection_score = round(0.55 * val_yt_f1 + 0.45 * val_f1, 2)
            else:
                val_yt_f1 = val_f1
                val_selection_score = val_f1

            scheduler.step(val_selection_score)

            history['train_loss'].append(round(train_loss, 4))
            history['val_loss'].append(round(val_loss, 4))
            history['train_acc'].append(round(train_acc, 2))
            history['val_acc'].append(round(val_acc, 2))
            history['val_f1'].append(round(val_f1, 2))
            history['val_yt_f1'].append(round(val_yt_f1, 2))
            history['val_rule_violation'].append(round(val_rv, 2))
            history['epoch_times'].append(round(ep_time, 2))

            improved = ""
            if val_selection_score > best_val_score:
                best_val_score = val_selection_score
                best_val_f1 = val_f1
                best_weights = {k: v.cpu().clone() for k, v in model.state_dict().items()}
                torch.save({
                    'model_key': model_key,
                    'epoch': global_epoch,
                    'best_val_f1': best_val_f1,
                    'best_val_yt_f1': val_yt_f1,
                    'best_val_score': best_val_score,
                    'model_state_dict': best_weights
                }, ckpt_path)
                improved = " [*BEST*]"
                patience_counter = 0
            else:
                patience_counter += 1

            print(
                f"Epoch {global_epoch:02d} (S{stage_idx}) | "
                f"Train Loss: {train_loss:.4f} Acc: {train_acc:5.2f}% | "
                f"Val F1: {val_f1:5.2f}% (YT-Val F1: {val_yt_f1:5.2f}%) | "
                f"RuleViol: {val_rv:4.1f}% | {ep_time:.1f}s{improved}"
            )

            cleanup_memory()
            if global_epoch >= 28 and patience_counter >= max_patience:
                print(f"[Early Stopping] Dừng sớm Stage {stage_idx} tại epoch {global_epoch} (đã hội tụ).")
                break

    if best_weights is not None:
        model.load_state_dict({k: v.to(DEVICE) for k, v in best_weights.items()})

    with open(hist_path, 'w', encoding='utf-8') as f:
        json.dump(history, f, indent=2)

    print(f"[OK] Đã lưu Checkpoint tốt nhất của {model_key} (Val Macro-F1 = {best_val_f1:.2f}%) tại: {ckpt_path}")
    return history

In [ ]:
# ==============================================================================
# BƯỚC 4.2: THỰC THI HUẤN LUYỆN 21 MÔ HÌNH ĐỐI CHỨNG (5 PHƯƠNG ÁN KIẾN TRÚC)
# ==============================================================================
# Cờ điều khiển: Đặt FORCE_RETRAIN = False để tự động nạp lại 19 mô hình đã có Checkpoint trên Drive và chỉ huấn luyện thêm 2 mô hình mới (M2b 1L, M2d 4L)
FORCE_RETRAIN = False

# Tiền nạp toàn bộ ảnh Mel & CQT vào RAM 1 lần duy nhất (~580 MB RAM) giúp mỗi epoch chạy trong ~2 giây
preload_all_spectrograms_to_ram(df_unified, img_size=IMG_SIZE)

# Khởi tạo DataLoaders cho 3 chế độ phổ: Đơn phổ Mel (1Ch), Đơn phổ CQT (1Ch) và Đa phổ Mel+CQT (2Ch)
loaders_map = {
    'mel': create_dataloaders(df_unified, spec_mode='mel', batch_size=BATCH_SIZE),
    'cqt': create_dataloaders(df_unified, spec_mode='cqt', batch_size=BATCH_SIZE),
    'mel+cqt': create_dataloaders(df_unified, spec_mode='mel+cqt', batch_size=BATCH_SIZE),
}
loaders_dual = loaders_map['mel+cqt']

EXPERIMENT_CONFIGS = [
    # ================= PHƯƠNG ÁN 1: BASELINE STANDARD CNN (3 MÔ HÌNH - 50 EPOCHS) =================
    {
        'family': '1_Baseline_CNN',
        'key': 'M1a_Baseline_CNN_Mel_1Ch',
        'display_name': 'M1a: Baseline CNN (Single Mel)',
        'spec_mode': 'mel',
        'builder': lambda: BaselineStandardCNN(in_channels=1, num_classes=6),
        'ep1': 50, 'ep2': 0, 'lr1': 1e-3, 'lr2': 0.0,
        'is_ns': False, 'l_con': 0.0, 'l_rule': 0.0
    },
    {
        'family': '1_Baseline_CNN',
        'key': 'M1b_Baseline_CNN_CQT_1Ch',
        'display_name': 'M1b: Baseline CNN (Single CQT)',
        'spec_mode': 'cqt',
        'builder': lambda: BaselineStandardCNN(in_channels=1, num_classes=6),
        'ep1': 50, 'ep2': 0, 'lr1': 1e-3, 'lr2': 0.0,
        'is_ns': False, 'l_con': 0.0, 'l_rule': 0.0
    },
    {
        'family': '1_Baseline_CNN',
        'key': 'M1c_Baseline_CNN_MelCQT_2Ch',
        'display_name': 'M1c: Baseline CNN (Mel+CQT)',
        'spec_mode': 'mel+cqt',
        'builder': lambda: BaselineStandardCNN(in_channels=2, num_classes=6),
        'ep1': 50, 'ep2': 0, 'lr1': 1e-3, 'lr2': 0.0,
        'is_ns': False, 'l_con': 0.0, 'l_rule': 0.0
    },

    # ================= PHƯƠNG ÁN 2: CNN + ATTENTION / TRANSFORMER (5 MÔ HÌNH - 50 EPOCHS) =================
    {
        'family': '2_CNN_Transformer',
        'key': 'M2a_CNN_Transformer_Mel_1Ch',
        'display_name': 'M2a: CNN + Transformer (Single Mel)',
        'spec_mode': 'mel',
        'builder': lambda: CNNTransformerNet(in_channels=1, num_classes=6, d_model=128, nhead=4, num_layers=2, use_se=False),
        'ep1': 50, 'ep2': 0, 'lr1': 1e-3, 'lr2': 0.0,
        'is_ns': False, 'l_con': 0.0, 'l_rule': 0.0
    },
    {
        'family': '2_CNN_Transformer',
        'key': 'M2b_CNN_Transformer_MelCQT_1L',
        'display_name': 'M2b: CNN + Transformer (Mel+CQT)',
        'spec_mode': 'mel+cqt',
        'builder': lambda: CNNTransformerNet(in_channels=2, num_classes=6, d_model=128, nhead=4, num_layers=1, use_se=False),
        'ep1': 50, 'ep2': 0, 'lr1': 1e-3, 'lr2': 0.0,
        'is_ns': False, 'l_con': 0.0, 'l_rule': 0.0
    },
    {
        'family': '2_CNN_Transformer',
        'key': 'M2c_CNN_Transformer_MelCQT_2L',
        'legacy_key': 'M2b_CNN_Transformer_MelCQT_2L',
        'display_name': 'M2c: CNN + Transformer 2L (Mel+CQT)',
        'spec_mode': 'mel+cqt',
        'builder': lambda: CNNTransformerNet(in_channels=2, num_classes=6, d_model=128, nhead=4, num_layers=2, use_se=False),
        'ep1': 50, 'ep2': 0, 'lr1': 1e-3, 'lr2': 0.0,
        'is_ns': False, 'l_con': 0.0, 'l_rule': 0.0
    },
    {
        'family': '2_CNN_Transformer',
        'key': 'M2d_CNN_Transformer_MelCQT_4L',
        'display_name': 'M2d: CNN + Transformer 4L (Mel+CQT)',
        'spec_mode': 'mel+cqt',
        'builder': lambda: CNNTransformerNet(in_channels=2, num_classes=6, d_model=128, nhead=4, num_layers=4, use_se=False),
        'ep1': 50, 'ep2': 0, 'lr1': 1e-3, 'lr2': 0.0,
        'is_ns': False, 'l_con': 0.0, 'l_rule': 0.0
    },
    {
        'family': '2_CNN_Transformer',
        'key': 'M2e_SE_CNN_Transformer_MelCQT_4L',
        'legacy_key': 'M2c_SE_CNN_Transformer_MelCQT_4L',
        'display_name': 'M2e: SE-CNN + Transformer 4L (Mel+CQT)',
        'spec_mode': 'mel+cqt',
        'builder': lambda: CNNTransformerNet(in_channels=2, num_classes=6, d_model=128, nhead=8, num_layers=4, use_se=True),
        'ep1': 50, 'ep2': 0, 'lr1': 1e-3, 'lr2': 0.0,
        'is_ns': False, 'l_con': 0.0, 'l_rule': 0.0
    },

    # ================= PHƯƠNG ÁN 3: PRETRAINED BACKBONE + TRANSFORMER (3 MÔ HÌNH - 50 EPOCHS) =================
    {
        'family': '3_Pretrained_Transformer',
        'key': 'M3a_Pretrained_ResNet34_Transformer',
        'display_name': 'M3a: Pretrained ResNet34 + Transformer',
        'spec_mode': 'mel+cqt',
        'builder': lambda: PretrainedTransformerNet(in_channels=2, num_classes=6, backbone_name='resnet34', d_model=128),
        'ep1': 10, 'ep2': 40, 'lr1': 1e-3, 'lr2': 2e-4,
        'is_ns': False, 'l_con': 0.0, 'l_rule': 0.0
    },
    {
        'family': '3_Pretrained_Transformer',
        'key': 'M3b_Pretrained_EfficientNetB0_Transformer',
        'display_name': 'M3b: Pretrained EfficientNet-B0 + Transformer',
        'spec_mode': 'mel+cqt',
        'builder': lambda: PretrainedTransformerNet(in_channels=2, num_classes=6, backbone_name='efficientnet_b0', d_model=128),
        'ep1': 10, 'ep2': 40, 'lr1': 1e-3, 'lr2': 2e-4,
        'is_ns': False, 'l_con': 0.0, 'l_rule': 0.0
    },
    {
        'family': '3_Pretrained_Transformer',
        'key': 'M3c_Pretrained_DenseNet121_Transformer',
        'display_name': 'M3c: Pretrained DenseNet121 + Transformer',
        'spec_mode': 'mel+cqt',
        'builder': lambda: PretrainedTransformerNet(in_channels=2, num_classes=6, backbone_name='densenet121', d_model=128),
        'ep1': 10, 'ep2': 40, 'lr1': 1e-3, 'lr2': 2e-4,
        'is_ns': False, 'l_con': 0.0, 'l_rule': 0.0
    },

    # ================= PHƯƠNG ÁN 4: MULTIMODAL FEATURE FUSION (5 MÔ HÌNH - 50 EPOCHS) =================
    {
        'family': '4_Multimodal_Fusion',
        'key': 'M4a_Multimodal_CNNTransformer_Concat',
        'display_name': 'M4a: Multimodal (CNN-Trans + 7 Axes Concat)',
        'spec_mode': 'mel+cqt',
        'builder': lambda: MultimodalFusionNet(in_channels=2, num_classes=6, acoustic_dim=7, spec_backbone='cnn_transformer', fusion_mode='concat'),
        'ep1': 50, 'ep2': 0, 'lr1': 1e-3, 'lr2': 0.0,
        'is_ns': False, 'l_con': 0.0, 'l_rule': 0.0
    },
    {
        'family': '4_Multimodal_Fusion',
        'key': 'M4b_Multimodal_ResNet34_Concat',
        'display_name': 'M4b: Multimodal (ResNet34 + 7 Axes Concat)',
        'spec_mode': 'mel+cqt',
        'builder': lambda: MultimodalFusionNet(in_channels=2, num_classes=6, acoustic_dim=7, spec_backbone='resnet34', fusion_mode='concat'),
        'ep1': 10, 'ep2': 40, 'lr1': 1e-3, 'lr2': 2e-4,
        'is_ns': False, 'l_con': 0.0, 'l_rule': 0.0
    },
    {
        'family': '4_Multimodal_Fusion',
        'key': 'M4c_Multimodal_EfficientNetB0_Gated',
        'display_name': 'M4c: Multimodal (EffNetB0 + 7 Axes Gated)',
        'spec_mode': 'mel+cqt',
        'builder': lambda: MultimodalFusionNet(in_channels=2, num_classes=6, acoustic_dim=7, spec_backbone='efficientnet_b0', fusion_mode='gated'),
        'ep1': 10, 'ep2': 40, 'lr1': 1e-3, 'lr2': 2e-4,
        'is_ns': False, 'l_con': 0.0, 'l_rule': 0.0
    },
    {
        'family': '4_Multimodal_Fusion',
        'key': 'M4d_Multimodal_EfficientNetB0_Concat',
        'display_name': 'M4d: Multimodal (EffNetB0 + 7 Axes Concat)',
        'spec_mode': 'mel+cqt',
        'builder': lambda: MultimodalFusionNet(in_channels=2, num_classes=6, acoustic_dim=7, spec_backbone='efficientnet_b0', fusion_mode='concat'),
        'ep1': 10, 'ep2': 40, 'lr1': 1e-3, 'lr2': 2e-4,
        'is_ns': False, 'l_con': 0.0, 'l_rule': 0.0
    },
    {
        'family': '4_Multimodal_Fusion',
        'key': 'M4e_Multimodal_DenseNet121_Concat',
        'display_name': 'M4e: Multimodal (DenseNet121 + 7 Axes Concat)',
        'spec_mode': 'mel+cqt',
        'builder': lambda: MultimodalFusionNet(in_channels=2, num_classes=6, acoustic_dim=7, spec_backbone='densenet121', fusion_mode='concat'),
        'ep1': 10, 'ep2': 40, 'lr1': 1e-3, 'lr2': 2e-4,
        'is_ns': False, 'l_con': 0.0, 'l_rule': 0.0
    },

    # ================= PHƯƠNG ÁN 5: NEURO-SYMBOLIC AI + FUZZY RULE LOSS (5 MÔ HÌNH - 50 EPOCHS, ĐỒNG BỘ 0.5 / 0.75) =================
    {
        'family': '5_NeuroSymbolic_AI',
        'key': 'M5a_NeuroSymbolic_CNNTransformer_Fuzzy',
        'display_name': 'M5a: Neuro-Symbolic (CNN-Trans + Fuzzy Loss)',
        'spec_mode': 'mel+cqt',
        'builder': lambda: NeuroSymbolicFuzzyNet(in_channels=2, num_classes=6, acoustic_dim=7, num_concepts=27, spec_backbone='cnn_transformer', fusion_mode='concat'),
        'ep1': 50, 'ep2': 0, 'lr1': 1e-3, 'lr2': 0.0,
        'is_ns': True, 'l_con': 0.5, 'l_rule': 0.75
    },
    {
        'family': '5_NeuroSymbolic_AI',
        'key': 'M5b_NeuroSymbolic_ResNet34_Fuzzy',
        'display_name': 'M5b: Neuro-Symbolic (ResNet34 + Fuzzy Loss)',
        'spec_mode': 'mel+cqt',
        'builder': lambda: NeuroSymbolicFuzzyNet(in_channels=2, num_classes=6, acoustic_dim=7, num_concepts=27, spec_backbone='resnet34', fusion_mode='concat'),
        'ep1': 10, 'ep2': 40, 'lr1': 1e-3, 'lr2': 2e-4,
        'is_ns': True, 'l_con': 0.5, 'l_rule': 0.75
    },
    {
        'family': '5_NeuroSymbolic_AI',
        'key': 'M5c_NeuroSymbolic_EfficientNetB0_GatedFuzzy',
        'display_name': 'M5c: Neuro-Symbolic (EffNetB0 Gated + Fuzzy Loss)',
        'spec_mode': 'mel+cqt',
        'builder': lambda: NeuroSymbolicFuzzyNet(in_channels=2, num_classes=6, acoustic_dim=7, num_concepts=27, spec_backbone='efficientnet_b0', fusion_mode='gated'),
        'ep1': 10, 'ep2': 40, 'lr1': 1e-3, 'lr2': 2e-4,
        'is_ns': True, 'l_con': 0.5, 'l_rule': 0.75
    },
    {
        'family': '5_NeuroSymbolic_AI',
        'key': 'M5d_NeuroSymbolic_EfficientNetB0_ConcatFuzzy',
        'display_name': 'M5d: Neuro-Symbolic (EffNetB0 Concat + Fuzzy Loss)',
        'spec_mode': 'mel+cqt',
        'builder': lambda: NeuroSymbolicFuzzyNet(in_channels=2, num_classes=6, acoustic_dim=7, num_concepts=27, spec_backbone='efficientnet_b0', fusion_mode='concat'),
        'ep1': 10, 'ep2': 40, 'lr1': 1e-3, 'lr2': 2e-4,
        'is_ns': True, 'l_con': 0.5, 'l_rule': 0.75
    },
    {
        'family': '5_NeuroSymbolic_AI',
        'key': 'M5e_NeuroSymbolic_DenseNet121_Fuzzy',
        'display_name': 'M5e: Neuro-Symbolic (DenseNet121 + Fuzzy Loss)',
        'spec_mode': 'mel+cqt',
        'builder': lambda: NeuroSymbolicFuzzyNet(in_channels=2, num_classes=6, acoustic_dim=7, num_concepts=27, spec_backbone='densenet121', fusion_mode='concat'),
        'ep1': 10, 'ep2': 40, 'lr1': 1e-3, 'lr2': 2e-4,
        'is_ns': True, 'l_con': 0.5, 'l_rule': 0.75
    },
]

trained_histories = {}
comparison_rows = []
confusion_data = {'in_domain': {}, 'youtube_zero_leakage': {}, 'unified_test': {}}

for cfg in EXPERIMENT_CONFIGS:
    m_key = cfg['key']
    m_name = cfg['display_name']
    m_family = cfg['family']
    loaders_curr = loaders_map[cfg['spec_mode']]

    model_instance = cfg['builder']().to(DEVICE)
    total_params, trainable_params = count_parameters(model_instance)

    hist = train_experiment_model(
        model_key=m_key,
        model=model_instance,
        loaders=loaders_curr,
        rule_engine=fuzzy_rule_criterion,
        epochs_stage1=cfg['ep1'],
        epochs_stage2=cfg['ep2'],
        lr_stage1=cfg['lr1'],
        lr_stage2=cfg['lr2'],
        is_neuro_symbolic=cfg['is_ns'],
        lambda_concept=cfg['l_con'],
        lambda_rule=cfg['l_rule'],
        force_retrain=FORCE_RETRAIN,
        legacy_key=cfg.get('legacy_key')
    )
    trained_histories[m_key] = {'name': m_name, 'family': m_family, 'history': hist}
    mean_ep_time = round(float(np.mean(hist['epoch_times'])), 2) if hist['epoch_times'] else 0.0

    # Đánh giá trên 3 kịch bản kiểm thử:
    # (1) In-Domain Test (Hugging Face - 720 mẫu phòng thu)
    res_hf = evaluate_model(model_instance, loaders_curr['test_hf'], fuzzy_rule_criterion, is_neuro_symbolic=cfg['is_ns'])
    confusion_data['in_domain'][m_name] = {'family': m_family, 'data': (res_hf['y_true'], res_hf['y_pred'])}

    # (2) Cross-Domain Zero-Leakage Test (YouTube - các video hoàn toàn độc lập)
    res_yt = evaluate_model(model_instance, loaders_curr['test_yt'], fuzzy_rule_criterion, is_neuro_symbolic=cfg['is_ns'])
    if len(res_yt['y_true']) > 0:
        confusion_data['youtube_zero_leakage'][m_name] = {'family': m_family, 'data': (res_yt['y_true'], res_yt['y_pred'])}

    # (3) Unified Combined Test (Tổng hợp Hugging Face + YouTube)
    res_all = evaluate_model(model_instance, loaders_curr['test_all'], fuzzy_rule_criterion, is_neuro_symbolic=cfg['is_ns'])
    confusion_data['unified_test'][m_name] = {'family': m_family, 'data': (res_all['y_true'], res_all['y_pred'])}

    for scenario_label, res_dict in [
        ('1_InDomain_HuggingFace_Test', res_hf),
        ('2_CrossDomain_YouTube_ZeroLeakage_Test', res_yt),
        ('3_Unified_Combined_Test', res_all)
    ]:
        if len(res_dict['y_true']) == 0:
            continue
        comparison_rows.append({
            'Model_Family': m_family,
            'Model_Key': m_key,
            'Model_Name': m_name,
            'Input_Mode': cfg['spec_mode'].upper(),
            'Test_Scenario': scenario_label,
            'Accuracy (%)': res_dict['acc'],
            'Balanced_Accuracy (%)': res_dict['bal_acc'],
            'Macro_Precision (%)': res_dict['macro_prec'],
            'Macro_Recall (%)': res_dict['macro_rec'],
            'Macro_F1 (%)': res_dict['macro_f1'],
            'Rule_Violation_Rate (%)': res_dict['rule_violation_rate'],
            'Epochs_Trained': len(hist['val_loss']),
            'Avg_Epoch_Time (s)': mean_ep_time,
            'Total_Params (M)': round(total_params / 1e6, 3)
        })

    # Lưu tạm bảng kết quả sau mỗi mô hình để an toàn tuyệt đối trên Google Drive
    pd.DataFrame(comparison_rows).to_csv(DIRS['results'] / 'model_comparison_metrics.csv', index=False, encoding='utf-8')
    del model_instance
    cleanup_memory()

---
## PHẦN 5: TỔNG HỢP BẢNG SO SÁNH 21 MÔ HÌNH, TRỰC QUAN HÓA BIỂU ĐỒ & SUY LUẬN XAI
Xuất toàn bộ bảng số liệu của 21 mô hình (5 Phương án kiến trúc) trên cả 3 kịch bản kiểm thử ra `experiments/results/model_comparison_metrics.csv` và lưu tự động các biểu đồ chất lượng cao (`.png`, 300 DPI) vào `experiments/plots/`.

In [ ]:
# ==============================================================================
# BƯỚC 5.1: XUẤT BẢNG SO SÁNH TỔNG HỢP 21 MÔ HÌNH RA CSV & HIỂN THỊ KẾT QUẢ
# ==============================================================================
df_metrics = pd.DataFrame(comparison_rows)
metrics_csv_path = DIRS['results'] / 'model_comparison_metrics.csv'
df_metrics.to_csv(metrics_csv_path, index=False, encoding='utf-8')

print("=" * 110)
print(f"[ĐÃ LƯU BẢNG KẾT QUẢ LUẬN VĂN ({df_metrics['Model_Key'].nunique()} MÔ HÌNH)] {metrics_csv_path}")
print("=" * 110)

for scenario in df_metrics['Test_Scenario'].unique():
    print(f"\n>>> KỊCH BẢN KIỂM THỬ: {scenario} <<<")
    cols_show = [
        'Model_Name', 'Input_Mode', 'Accuracy (%)', 'Balanced_Accuracy (%)',
        'Macro_F1 (%)', 'Rule_Violation_Rate (%)', 'Epochs_Trained', 'Avg_Epoch_Time (s)', 'Total_Params (M)'
    ]
    avail_cols = [c for c in cols_show if c in df_metrics.columns]
    sub_df = df_metrics[df_metrics['Test_Scenario'] == scenario][avail_cols]
    print(sub_df.to_string(index=False))

In [ ]:
# ==============================================================================
# BƯỚC 5.2: VẼ & LƯU TỰ ĐỘNG CÁC BIỂU ĐỒ GOM NHÓM THEO TỪNG PHƯƠNG ÁN (.PNG)
# ==============================================================================
plt.rcParams['font.sans-serif'] = ['DejaVu Sans', 'Arial', 'Segoe UI']
plt.rcParams['axes.unicode_minus'] = False

FAMILY_TITLES = {
    '1_Baseline_CNN': 'Phương án 1: Baseline CNN',
    '2_CNN_Transformer': 'Phương án 2: CNN + Transformer',
    '3_Pretrained_Transformer': 'Phương án 3: Pretrained + Trans',
    '4_Multimodal_Fusion': 'Phương án 4: Multimodal Fusion',
    '5_NeuroSymbolic_AI': 'Phương án 5: Neuro-Symbolic AI'
}
FAMILY_ORDER = list(FAMILY_TITLES.keys())

def format_short_model_label(full_name: str, multiline: bool = False) -> str:
    code_part, rest = full_name.split(':', 1) if ':' in full_name else (full_name, '')
    rest = rest.strip()
    for pref in ['Baseline CNN ', 'CNN + Transformer ', 'SE-CNN + Transformer ', 'Pretrained ', 'Multimodal ', 'Neuro-Symbolic ']:
        if full_name.startswith(code_part + ': SE-CNN'):
            if pref == 'SE-CNN + Transformer ':
                rest = rest.replace(pref, 'SE ')
        elif rest.startswith(pref):
            rest = rest[len(pref):]
            break
    rest = rest.replace('+ Transformer', 'Trans').replace(' + 7 Axes ', ' + ').strip()
    if rest.startswith('(') and rest.endswith(')'):
        rest_inner = rest[1:-1]
    else:
        rest_inner = rest
    if multiline:
        rest_inner = rest_inner.replace(' + ', '\n').replace(' (', '\n(')
        return f"{code_part}\n{rest_inner}"
    return f"{code_part}: {rest_inner}"

# 1. Biểu đồ Đường cong Huấn luyện GOM THEO TỪNG PHƯƠNG ÁN (5 hàng x 3 cột cực kỳ thoáng, không rối mắt)
fig, axes = plt.subplots(len(FAMILY_ORDER), 3, figsize=(21, 4.2 * len(FAMILY_ORDER)))
palette_per_row = ['#1f77b4', '#d62728', '#2ca02c', '#9467bd', '#ff7f0e']

for row_idx, fam_key in enumerate(FAMILY_ORDER):
    fam_items = [(k, v) for k, v in trained_histories.items() if v['family'] == fam_key]
    ax_loss, ax_f1, ax_rv = axes[row_idx, 0], axes[row_idx, 1], axes[row_idx, 2]

    for m_idx, (m_key, info) in enumerate(fam_items):
        h = info['history']
        lbl = format_short_model_label(info['name'], multiline=False)
        col = palette_per_row[m_idx % len(palette_per_row)]
        epochs_range = range(1, len(h['val_loss']) + 1)

        ax_loss.plot(epochs_range, h['train_loss'], linestyle='--', alpha=0.35, color=col)
        ax_loss.plot(epochs_range, h['val_loss'], linewidth=2.0, label=lbl, color=col)
        ax_f1.plot(epochs_range, h['val_f1'], linewidth=2.0, marker='o', markersize=2.5, label=lbl, color=col)
        ax_rv.plot(epochs_range, h['val_rule_violation'], linewidth=2.0, marker='s', markersize=2.5, label=lbl, color=col)

    fam_title = FAMILY_TITLES[fam_key]
    ax_loss.set_title(f"{fam_title} — Loss (Train -- / Val —)", fontweight='bold', fontsize=10.5)
    ax_loss.set_ylabel('Loss'); ax_loss.grid(alpha=0.3); ax_loss.legend(fontsize=8, loc='upper right')

    ax_f1.set_title(f"{fam_title} — Val Macro-F1 (%)", fontweight='bold', fontsize=10.5)
    ax_f1.set_ylabel('Macro-F1 (%)'); ax_f1.set_ylim(30, 100); ax_f1.grid(alpha=0.3); ax_f1.legend(fontsize=8, loc='lower right')

    ax_rv.set_title(f"{fam_title} — Val Rule Violation (%)", fontweight='bold', fontsize=10.5)
    ax_rv.set_ylabel('Rule Violation (%)'); ax_rv.set_ylim(20, 85); ax_rv.grid(alpha=0.3); ax_rv.legend(fontsize=8, loc='upper right')

    if row_idx == len(FAMILY_ORDER) - 1:
        ax_loss.set_xlabel('Epoch'); ax_f1.set_xlabel('Epoch'); ax_rv.set_xlabel('Epoch')

fig.suptitle('Quá trình Huấn luyện theo từng Phương án Kiến trúc (50 Epochs)', fontsize=14, fontweight='bold', y=1.005)
plt.tight_layout()
curves_plot_path = DIRS['plots'] / 'training_loss_f1_rule_curves.png'
plt.savefig(curves_plot_path, dpi=300, bbox_inches='tight')
plt.show()
print(f"[OK] Đã lưu biểu đồ quá trình huấn luyện gom theo phương án tại: {curves_plot_path}")

# 2. Hàm vẽ lưới Confusion Matrix GOM THEO TỪNG HÀNG PHƯƠNG ÁN (Mỗi hàng = 1 Phương án)
def plot_confusion_matrices_by_family(cm_dict: Dict[str, Dict[str, Any]], title: str, save_name: str):
    if not cm_dict:
        return
    fam_groups = {f: [] for f in FAMILY_ORDER}
    for m_name, payload in cm_dict.items():
        fam_groups.setdefault(payload['family'], []).append((m_name, payload['data']))

    max_cols = max((len(v) for v in fam_groups.values()), default=3)
    rows = len(FAMILY_ORDER)
    fig, axes = plt.subplots(rows, max_cols, figsize=(4.8 * max_cols, 4.6 * rows))

    for r_idx, fam_key in enumerate(FAMILY_ORDER):
        items = fam_groups.get(fam_key, [])
        for c_idx in range(max_cols):
            ax = axes[r_idx, c_idx]
            if c_idx >= len(items):
                ax.axis('off')
                continue
            m_name, (y_true, y_pred) = items[c_idx]
            cm = confusion_matrix(y_true, y_pred, labels=list(range(len(GENRES))))
            cm_norm = cm.astype('float') / np.maximum(cm.sum(axis=1, keepdims=True), 1) * 100.0
            sns.heatmap(
                cm_norm, annot=True, fmt='.1f', cmap='Blues',
                xticklabels=GENRES, yticklabels=GENRES, ax=ax, cbar=False, annot_kws={'size': 8}
            )
            acc_v = accuracy_score(y_true, y_pred) * 100.0
            f1_v = f1_score(y_true, y_pred, average='macro', zero_division=0) * 100.0
            short_title = m_name.replace(': Multimodal ', ': ').replace(': Neuro-Symbolic ', ': ').replace(': Pretrained ', ': ')
            ax.set_title(f"{short_title}\nAcc: {acc_v:.1f}% | F1: {f1_v:.1f}%", fontsize=9, fontweight='bold')
            if r_idx == rows - 1:
                ax.set_xlabel('Predicted', fontsize=8.5)
            if c_idx == 0:
                ax.set_ylabel(f"{FAMILY_TITLES[fam_key].split(':')[0]}\nTrue", fontsize=9, fontweight='bold')
            ax.tick_params(axis='x', rotation=30, labelsize=8)
            ax.tick_params(axis='y', rotation=0, labelsize=8)

    fig.suptitle(title, fontsize=14, fontweight='bold', y=1.005)
    plt.tight_layout()
    out_p = DIRS['plots'] / save_name
    plt.savefig(out_p, dpi=300, bbox_inches='tight')
    plt.show()
    print(f"[OK] Đã lưu Confusion Matrix gom theo phương án tại: {out_p}")

plot_confusion_matrices_by_family(
    confusion_data['in_domain'],
    title='Ma trận Nhầm lẫn Chuẩn hóa (%) theo Phương án — Kịch bản 1: In-Domain Test (Hugging Face)',
    save_name='confusion_matrices_in_domain_test.png'
)

if confusion_data['youtube_zero_leakage']:
    plot_confusion_matrices_by_family(
        confusion_data['youtube_zero_leakage'],
        title='Ma trận Nhầm lẫn Chuẩn hóa (%) theo Phương án — Kịch bản 2: Cross-Domain Zero-Leakage Test (YouTube)',
        save_name='confusion_matrices_youtube_zero_leakage.png'
    )

# 3. Biểu đồ cột ĐỐI SÁNH CHIA THEO 5 PHƯƠNG ÁN (2 hàng Kịch bản x 5 cột Phương án -> chữ to, số không bao giờ đè nhau)
scenarios_to_plot = [
    ('2_CrossDomain_YouTube_ZeroLeakage_Test', 'Kịch bản 2: Cross-Domain Zero-Leakage (YouTube Độc lập)'),
    ('3_Unified_Combined_Test', 'Kịch bản 3: Unified Combined Test (Hugging Face + YouTube)')
]
fig, axes = plt.subplots(2, len(FAMILY_ORDER), figsize=(25, 10.5), sharey=True)

for row_idx, (scen_key, scen_title) in enumerate(scenarios_to_plot):
    df_scen = df_metrics[df_metrics['Test_Scenario'] == scen_key].copy()
    for col_idx, fam_key in enumerate(FAMILY_ORDER):
        ax = axes[row_idx, col_idx]
        df_fam = df_scen[df_scen['Model_Family'] == fam_key].copy()
        if df_fam.empty:
            ax.axis('off')
            continue

        x_pos = np.arange(len(df_fam))
        width = 0.36
        b1 = ax.bar(x_pos - width/2, df_fam['Macro_F1 (%)'], width, label='Macro-F1 (%) [Cao = Tốt]', color='#2E86AB', edgecolor='#1B4965')
        b2 = ax.bar(x_pos + width/2, df_fam['Rule_Violation_Rate (%)'], width, label='Rule Violation (%) [Thấp = Tốt]', color='#E63946', edgecolor='#9D0208')

        short_labels = [format_short_model_label(n, multiline=True) for n in df_fam['Model_Name']]

        ax.set_xticks(x_pos)
        ax.set_xticklabels(short_labels, fontsize=7.8, rotation=18, ha='right')
        ax.set_ylim(0, 105)
        ax.grid(axis='y', alpha=0.3)

        if row_idx == 0:
            ax.set_title(FAMILY_TITLES[fam_key], fontweight='bold', fontsize=10.5, pad=8)
        if col_idx == 0:
            ax.set_ylabel(f"{scen_title.split(':')[0]}\nTỷ lệ (%)", fontweight='bold', fontsize=9.5)
        if row_idx == 0 and col_idx == len(FAMILY_ORDER) - 1:
            ax.legend(loc='upper right', fontsize=8)

        for bar in b1:
            ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1.2, f"{bar.get_height():.1f}", ha='center', fontsize=7.5, fontweight='bold', color='#1B4965')
        for bar in b2:
            ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1.2, f"{bar.get_height():.1f}", ha='center', fontsize=7.5, fontweight='bold', color='#9D0208')

fig.suptitle('Đối sánh Macro-F1 & Tỷ lệ Vi phạm Luật mờ theo từng Phương án Kiến trúc', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
bar_path = DIRS['plots'] / 'benchmark_macro_f1_vs_rule_violation.png'
plt.savefig(bar_path, dpi=300, bbox_inches='tight')
plt.show()
print(f"[OK] Đã lưu biểu đồ đối sánh chia theo 5 phương án tại: {bar_path}")

# 4. Biểu đồ Tổng quan Tóm tắt 5 Phương án (Lấy mô hình tốt nhất của mỗi Phương án qua 3 Kịch bản)
fig, ax = plt.subplots(figsize=(12, 6))
scen_list = [
    ('1_InDomain_HuggingFace_Test', '1. In-Domain (Hugging Face)', '#457B9D'),
    ('2_CrossDomain_YouTube_ZeroLeakage_Test', '2. Cross-Domain (YouTube)', '#E63946'),
    ('3_Unified_Combined_Test', '3. Unified Combined', '#2A9D8F')
]
x_fam = np.arange(len(FAMILY_ORDER))
w = 0.25

for s_idx, (s_key, s_lbl, s_col) in enumerate(scen_list):
    best_vals = []
    for fam_key in FAMILY_ORDER:
        sub = df_metrics[(df_metrics['Test_Scenario'] == s_key) & (df_metrics['Model_Family'] == fam_key)]
        best_vals.append(float(sub['Macro_F1 (%)'].max()) if not sub.empty else 0.0)
    bars = ax.bar(x_fam + (s_idx - 1) * w, best_vals, w, label=s_lbl, color=s_col, edgecolor='black', linewidth=0.5)
    for b in bars:
        ax.text(b.get_x() + b.get_width()/2, b.get_height() + 1.0, f"{b.get_height():.1f}%", ha='center', fontsize=8.5, fontweight='bold')

ax.set_xticks(x_fam)
ax.set_xticklabels([FAMILY_TITLES[f].replace(': ', '\n') for f in FAMILY_ORDER], fontsize=9.5, fontweight='bold')
ax.set_ylabel('Best Macro-F1 (%)', fontweight='bold')
ax.set_ylim(0, 108)
ax.set_title('Tổng quan Hiệu năng Đỉnh (Best Macro-F1) giữa 5 Phương án trên 3 Kịch bản Kiểm thử', fontweight='bold', fontsize=12.5)
ax.legend(loc='upper right')
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
fam_summary_path = DIRS['plots'] / 'benchmark_family_comparison.png'
plt.savefig(fam_summary_path, dpi=300, bbox_inches='tight')
plt.show()
print(f"[OK] Đã lưu biểu đồ tổng quan 5 phương án tại: {fam_summary_path}")

In [ ]:
# ==============================================================================
# BƯỚC 5.3: MINH HỌA GIẢI THÍCH CỤC BỘ (NEURO-SYMBOLIC XAI) KẾT HỢP AUDIO CONCEPTS
# ==============================================================================
def load_vietnamese_audio_concepts(md_path: Path) -> Dict[str, List[str]]:
    """Đọc 42 câu giải thích Audio Concepts tiếng Việt từ concept/audio_concepts.md."""
    genre_map_code = {
        'CL': 'Cải lương', 'CT': 'Ca trù', 'CH': 'Chèo',
        'CV': 'Chầu văn', 'QH': 'Quan họ', 'HX': 'Hát xẩm'
    }
    concepts_by_genre = {g: [] for g in GENRES}
    if not md_path.exists():
        return concepts_by_genre
    with open(md_path, 'r', encoding='utf-8') as f:
        for line in f:
            m = re.search(r'`((?:CL|CT|CH|CV|QH|HX)_AUD_\d+)`\s*\|\s*`[^`]+`\s*\|\s*`[^`]+`\s*\|\s*([^|]+)\|', line)
            if m:
                cid, sentence = m.group(1).strip(), m.group(2).strip()
                prefix = cid.split('_')[0]
                g_name = genre_map_code.get(prefix)
                if g_name:
                    concepts_by_genre[g_name].append(f"[{cid}] {sentence}")
    return concepts_by_genre

audio_concepts_dict = load_vietnamese_audio_concepts(DIRS['concept'] / 'audio_concepts.md')

def explain_single_sample_neuro_symbolic(sample_index: int = 0):
    """So sánh dự đoán và sinh lời giải thích Neuro-Symbolic cho 1 mẫu kiểm thử."""
    ds_test = loaders_dual['test_all'].dataset
    if len(ds_test) == 0:
        return
    idx = min(sample_index, len(ds_test) - 1)
    sample_tuple = ds_test[idx]
    spec_t, acoustic_t, fuzzy_gt, label_t = sample_tuple[0], sample_tuple[1], sample_tuple[2], sample_tuple[3]

    m1 = BaselineStandardCNN(in_channels=2, num_classes=6).to(DEVICE)
    m5 = NeuroSymbolicFuzzyNet(in_channels=2, num_classes=6, acoustic_dim=7, num_concepts=27, spec_backbone='cnn_transformer', fusion_mode='concat').to(DEVICE)

    p1_candidates = [
        DIRS['checkpoints'] / 'M1c_Baseline_CNN_MelCQT_2Ch_best.pth',
        DIRS['checkpoints'] / 'M1b_Baseline_CNN_MelCQT_2Ch_best.pth'
    ]
    p5_candidates = [
        DIRS['checkpoints'] / 'M5a_NeuroSymbolic_CNNTransformer_Fuzzy_best.pth',
        DIRS['checkpoints'] / 'M5_NeuroSymbolic_FuzzyRuleLoss_best.pth'
    ]
    p1 = next((p for p in p1_candidates if p.exists()), None)
    p5 = next((p for p in p5_candidates if p.exists()), None)

    if p1 is not None:
        m1.load_state_dict(torch.load(p1, map_location=DEVICE, weights_only=False)['model_state_dict'])
    if p5 is not None:
        m5.load_state_dict(torch.load(p5, map_location=DEVICE, weights_only=False)['model_state_dict'])

    m1.eval(); m5.eval()
    with torch.no_grad():
        inp_s = spec_t.unsqueeze(0).to(DEVICE)
        inp_a = acoustic_t.unsqueeze(0).to(DEVICE)
        prob_m1 = torch.softmax(m1(inp_s, inp_a), dim=-1)[0].cpu().numpy()
        logits_m5, concepts_m5 = m5(inp_s, inp_a, return_concepts=True)
        prob_m5 = torch.softmax(logits_m5, dim=-1)[0].cpu().numpy()
        rule_acts = fuzzy_rule_criterion.compute_rule_activations(concepts_m5)

    true_genre = IDX_TO_GENRE[int(label_t.item())]
    pred_m1 = IDX_TO_GENRE[int(np.argmax(prob_m1))]
    pred_m5 = IDX_TO_GENRE[int(np.argmax(prob_m5))]

    print("=" * 95)
    print(f"MINH HỌA SUY LUẬN & GIẢI THÍCH NEURO-SYMBOLIC XAI (Mẫu kiểm thử #{idx})")
    print("=" * 95)
    print(f"  + Nhãn thực tế (Ground Truth)      : {true_genre}")
    print(f"  + Dự đoán Baseline Standard CNN    : {pred_m1} ({prob_m1.max()*100:.1f}%)")
    print(f"  + Dự đoán Neuro-Symbolic Model (M5): {pred_m5} ({prob_m5.max()*100:.1f}%)")
    print("\n  [Vector 7 trục âm học MPEG-7 chuẩn hóa]:")
    for ax_name, val in zip(AXES_7_COLS, acoustic_t.numpy()):
        print(f"    - {ax_name:<10}: {val:.3f}")

    scored_rules = []
    for r_info, (firing_t, _, conf) in zip(fuzzy_rule_criterion.parsed_rules, rule_acts):
        f_val = float(firing_t[0].item())
        if f_val > 0.30:
            scored_rules.append((f_val * conf, f_val, r_info))
    scored_rules.sort(key=lambda x: x[0], reverse=True)

    print("\n  [Các Luật mờ IF-THEN kích hoạt mạnh nhất từ rules_decision_tree_continuous.json]:")
    for _, f_val, r_info in scored_rules[:3]:
        print(f"    * IF ({r_info['rule_text']}) => THEN {r_info['consequent_name']} "
              f"[Độ kích hoạt: {f_val:.2f} | Độ tin cậy luật: {r_info['confidence']:.2f}]")

    print(f"\n  [Diễn giải Ngữ nghĩa Âm học (Atomic Audio Concepts cho {pred_m5})]:")
    for c_sent in audio_concepts_dict.get(pred_m5, [])[:3]:
        print(f"    - {c_sent}")
    print("=" * 95)

explain_single_sample_neuro_symbolic(sample_index=0)